[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/10_matrix_calculus_graph_and_ai_applications/exercises.ipynb)

# Module 10 — Exercises: Matrix Calculus, Graphs, and AI Applications

Forty-eight solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it.

Theorem and proof numbers refer to [first_principles.ipynb](first_principles.ipynb). Symbols
follow [the notation register](../../docs/notation.md): transition matrices are
column-stochastic with $P\pi = \pi$, norms are written $\lVert x \rVert$, and Laplacian spectra
are listed ascending, $0 = \lambda_1 \le \lambda_2 \le \cdots \le \lambda_n$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
vec = lambda M: M.reshape(-1, order="F")
print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Shape of a matrix gradient

**Statement.** For $f : \mathbb{R}^{m \times n} \to \mathbb{R}$ differentiable at $X$, what is the
shape of $\nabla_X f$, and what is the shape of the Jacobian of $F : \mathbb{R}^n \to \mathbb{R}^m$?

**Intuition.** A gradient has to be added to its variable during a gradient step, so it must have
the variable's shape.

**Solution.**

*Step 1.* Theorem 4.1 writes $df = \operatorname{tr}(G^{\top}dX)$, and the trace is defined only
when $G^{\top}dX$ is square, which forces $G \in \mathbb{R}^{m \times n}$.

*Step 2.* A Jacobian maps input perturbations to output perturbations, so $J_F \in \mathbb{R}^{m \times n}$
with $(J_F)_{ij} = \partial F_i / \partial x_j$; for $m = 1$ it is the row vector $(\nabla F)^{\top}$.

$$
\boxed{\nabla_X f \in \mathbb{R}^{m \times n}, \qquad J_F \in \mathbb{R}^{m \times n}, \qquad J_f = (\nabla f)^{\top} \text{ when } m = 1}
$$

**Key takeaway.** Shape checking is the cheapest error detector in matrix calculus: a formula
whose two sides have different shapes is wrong before any arithmetic happens.

In [2]:
m, n = 3, 4
X = rng.standard_normal((m, n))
Afix = rng.standard_normal((m, n))
f = lambda Z: np.sum(Afix * Z)          # f(X) = tr(A^T X), gradient A
G = Afix
print("X shape        :", X.shape)
print("gradient shape :", G.shape)
h = 1e-6
fd = np.array([[(f(X + h * np.eye(1, m * n, i * n + j).reshape(m, n))
                 - f(X - h * np.eye(1, m * n, i * n + j).reshape(m, n))) / (2 * h)
                for j in range(n)] for i in range(m)])
print("finite-difference gradient shape:", fd.shape, " max error", np.abs(fd - G).max())
assert G.shape == X.shape and np.abs(fd - G).max() < 1e-8

X shape        : (3, 4)
gradient shape : (3, 4)
finite-difference gradient shape: (3, 4)  max error 1.7385204387210251e-10


### Problem L0.2 — The Frobenius inner product as a trace

**Statement.** Verify $\langle A, B \rangle_F = \operatorname{tr}(A^{\top}B) = \sum_{ij}A_{ij}B_{ij}$
for $A = \left[\begin{smallmatrix}1&2\\3&4\end{smallmatrix}\right]$ and
$B = \left[\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right]$.

**Intuition.** The trace of $A^{\top}B$ sums the diagonal, and the $i$-th diagonal entry is the
dot product of the $i$-th columns.

**Solution.**

*Step 1.* Entrywise: $1 \cdot 0 + 2 \cdot 1 + 3 \cdot 1 + 4 \cdot 0 = 5$.

*Step 2.* By trace: $A^{\top}B = \left[\begin{smallmatrix}1&3\\2&4\end{smallmatrix}\right]\left[\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right] = \left[\begin{smallmatrix}3&1\\4&2\end{smallmatrix}\right]$,
whose trace is $3 + 2 = 5$.

$$
\boxed{\langle A, B \rangle_F = 5}
$$

**Key takeaway.** This inner product is the one Theorem 4.1 represents differentials in, so every
matrix gradient in this module is measured against it.

In [3]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[0.0, 1.0], [1.0, 0.0]])
print("entrywise sum :", np.sum(A * B))
print("trace(A^T B)  :", np.trace(A.T @ B))
assert np.isclose(np.sum(A * B), 5.0) and np.isclose(np.trace(A.T @ B), 5.0)

entrywise sum : 5.0
trace(A^T B)  : 5.0


### Problem L0.3 — Gradient of a linear form and of the squared norm

**Statement.** Compute $\nabla_x (a^{\top}x)$ and $\nabla_x \lVert x \rVert_2^2$ for
$x, a \in \mathbb{R}^n$.

**Intuition.** These are the vector versions of $\tfrac{d}{dx}(ax) = a$ and
$\tfrac{d}{dx}(x^2) = 2x$.

**Solution.**

*Step 1.* $d(a^{\top}x) = a^{\top}dx$, so $\nabla_x(a^{\top}x) = a$ by Theorem 4.1.

*Step 2.* $d(x^{\top}x) = (dx)^{\top}x + x^{\top}dx = 2x^{\top}dx$, so
$\nabla_x \lVert x \rVert_2^2 = 2x$.

$$
\boxed{\nabla_x (a^{\top}x) = a, \qquad \nabla_x \lVert x \rVert_2^2 = 2x}
$$

**Key takeaway.** Linear terms have constant gradients; quadratic terms have gradients linear in
the variable. Every loss in this module is built from these two pieces.

In [4]:
a = np.array([1.0, -2.0, 0.5])
x = np.array([0.3, 1.1, -0.7])
h = 1e-6
fd_lin = np.array([( (a @ (x + h * np.eye(3)[i])) - (a @ (x - h * np.eye(3)[i])) ) / (2 * h)
                   for i in range(3)])
fd_sq = np.array([((x + h * np.eye(3)[i]) @ (x + h * np.eye(3)[i])
                   - (x - h * np.eye(3)[i]) @ (x - h * np.eye(3)[i])) / (2 * h) for i in range(3)])
print("analytic a  :", a, "  finite difference:", fd_lin)
print("analytic 2x :", 2 * x, "  finite difference:", fd_sq)
assert np.abs(fd_lin - a).max() < 1e-8 and np.abs(fd_sq - 2 * x).max() < 1e-8

analytic a  : [ 1.  -2.   0.5]   finite difference: [ 1.  -2.   0.5]
analytic 2x : [ 0.6  2.2 -1.4]   finite difference: [ 0.6  2.2 -1.4]


### Problem L0.4 — Matrix exponential of a diagonal matrix

**Statement.** Compute $e^{At}$ for $A = \operatorname{diag}(\lambda_1, \dots, \lambda_n)$.

**Intuition.** A diagonal matrix decouples the coordinates completely, so each one evolves as an
independent scalar exponential.

**Solution.**

*Step 1.* $A^k = \operatorname{diag}(\lambda_1^k, \dots, \lambda_n^k)$, because powers of a
diagonal matrix act entry by entry.

*Step 2.* The series of Definition 3.6 is therefore diagonal, with $i$-th entry
$\sum_k (\lambda_i t)^k / k! = e^{\lambda_i t}$.

$$
\boxed{e^{At} = \operatorname{diag}\bigl(e^{\lambda_1 t}, \dots, e^{\lambda_n t}\bigr)}
$$

**Key takeaway.** Diagonalizing $A$ reduces $e^{At}$ to this case, which is why the spectrum
controls the dynamics of $\dot{x} = Ax$.

In [5]:
lam = np.array([1.0, -2.0, 0.5])
t = 0.8
print("expm(diag(lam) t):\n", sla.expm(np.diag(lam) * t))
print("diag(exp(lam t)) :\n", np.diag(np.exp(lam * t)))
assert np.allclose(sla.expm(np.diag(lam) * t), np.diag(np.exp(lam * t)))

expm(diag(lam) t):
 [[2.2255 0.     0.    ]
 [0.     0.2019 0.    ]
 [0.     0.     1.4918]]
diag(exp(lam t)) :
 [[2.2255 0.     0.    ]
 [0.     0.2019 0.    ]
 [0.     0.     1.4918]]


### Problem L0.5 — Exponential of a nilpotent matrix

**Statement.** Let $N \in \mathbb{R}^{n \times n}$ satisfy $N^2 = 0$. Compute $e^{N}$.

**Intuition.** The exponential series stops as soon as the powers vanish.

**Solution.**

*Step 1.* $N^k = 0$ for all $k \ge 2$.

*Step 2.* The series of Definition 3.6 truncates after two terms.

$$
\boxed{e^{N} = I + N}
$$

**Key takeaway.** Nilpotent parts of a matrix contribute polynomials, not exponentials — the
mechanism behind the $t e^{\lambda t}$ term of Example 6.5.

In [6]:
N = np.array([[0.0, 3.0], [0.0, 0.0]])
print("N^2 =\n", N @ N)
print("expm(N) =\n", sla.expm(N))
print("I + N  =\n", np.eye(2) + N)
assert np.allclose(sla.expm(N), np.eye(2) + N)

N^2 =
 [[0. 0.]
 [0. 0.]]
expm(N) =
 [[1. 3.]
 [0. 1.]]
I + N  =
 [[1. 3.]
 [0. 1.]]


### Problem L0.6 — Why $e^{A}$ is not the entrywise exponential

**Statement.** For $A = \left[\begin{smallmatrix}0&1\\0&0\end{smallmatrix}\right]$, compare
$e^{A}$ with the matrix of entrywise exponentials $[e^{A_{ij}}]$.

**Intuition.** The series involves products of $A$ with itself, which mix entries; the entrywise
exponential never does.

**Solution.**

*Step 1.* $A^2 = 0$, so $e^{A} = I + A = \left[\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right]$
by Problem L0.5.

*Step 2.* Entrywise, $[e^{A_{ij}}] = \left[\begin{smallmatrix}1&e\\1&1\end{smallmatrix}\right]$,
which is not even close: it has no zero entry at all.

$$
\boxed{e^{A} = \begin{pmatrix}1&1\\0&1\end{pmatrix} \neq \begin{pmatrix}1&e\\1&1\end{pmatrix} = [e^{A_{ij}}]}
$$

**Key takeaway.** `numpy.exp(A)` is the entrywise exponential and `scipy.linalg.expm(A)` is the
matrix exponential. Confusing them is a silent, wrong answer.

In [7]:
A = np.array([[0.0, 1.0], [0.0, 0.0]])
print("scipy.linalg.expm(A):\n", sla.expm(A))
print("numpy.exp(A)        :\n", np.exp(A))
print("difference in Frobenius norm:", np.linalg.norm(sla.expm(A) - np.exp(A)))
assert np.linalg.norm(sla.expm(A) - np.exp(A)) > 1.0

scipy.linalg.expm(A):
 [[1. 1.]
 [0. 1.]]
numpy.exp(A)        :
 [[1.     2.7183]
 [1.     1.    ]]
difference in Frobenius norm: 1.9880876343895304


### Problem L0.7 — Every Laplacian kills the constant vector

**Statement.** Show that $L\mathbf{1} = 0$ for every graph Laplacian $L = D - A$.

**Intuition.** A constant signal has no disagreement across any edge, so it has zero Dirichlet
energy.

**Solution.**

*Step 1.* The $i$-th entry of $A\mathbf{1}$ is $\sum_j A_{ij} = d_i$.

*Step 2.* The $i$-th entry of $D\mathbf{1}$ is also $d_i$, so the difference is zero.

$$
\boxed{L\mathbf{1} = D\mathbf{1} - A\mathbf{1} = d - d = 0}
$$

**Key takeaway.** $\lambda_1 = 0$ is not an accident of a particular graph; it is forced by the
definition, and it is why Theorem 4.6 looks at $\lambda_2$.

In [8]:
Ag = np.array([[0.0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]])
Lg = np.diag(Ag.sum(1)) - Ag
print("L 1 =", Lg @ np.ones(4))
print("smallest eigenvalue:", np.linalg.eigvalsh(Lg)[0])
assert np.allclose(Lg @ np.ones(4), 0.0)

L 1 = [0. 0. 0. 0.]
smallest eigenvalue: -1.7752861384697267e-16


### Problem L0.8 — The trace of a Laplacian counts edges

**Statement.** For an unweighted graph, express $\operatorname{tr}(L)$ in terms of
$\lvert E \rvert$.

**Intuition.** Each edge contributes $1$ to the degree of each of its two endpoints.

**Solution.**

*Step 1.* $\operatorname{tr}(L) = \sum_i (d_i - A_{ii}) = \sum_i d_i$, since $A_{ii} = 0$.

*Step 2.* The handshake identity gives $\sum_i d_i = 2\lvert E \rvert$.

$$
\boxed{\operatorname{tr}(L) = \sum_i \lambda_i = 2\lvert E \rvert}
$$

**Key takeaway.** The Laplacian spectrum knows the edge count without knowing the graph — the
first term of the heat-kernel expansion in Problem L2.14.

In [9]:
edges = [(i, j) for i in range(4) for j in range(i + 1, 4) if Ag[i, j] > 0]
print("edges:", edges, " |E| =", len(edges))
print("trace(L) =", np.trace(Lg), "  sum of eigenvalues =", np.linalg.eigvalsh(Lg).sum())
assert np.isclose(np.trace(Lg), 2 * len(edges))

edges: [(0, 1), (0, 2), (1, 2), (2, 3)]  |E| = 4
trace(L) = 8.0   sum of eigenvalues = 8.0


### Problem L0.9 — vec and the Kronecker product on a $2 \times 2$ example

**Statement.** With $A = \left[\begin{smallmatrix}1&2\\3&4\end{smallmatrix}\right]$ and
$X = \left[\begin{smallmatrix}1&0\\0&1\end{smallmatrix}\right]$, check
$\operatorname{vec}(AX) = (I \otimes A)\operatorname{vec}(X)$.

**Intuition.** With $B = I$ the identity of Theorem 4.3 says that left multiplication acts on each
column of $X$ separately.

**Solution.**

*Step 1.* $AX = A$, so $\operatorname{vec}(AX) = (1, 3, 2, 4)^{\top}$ stacking columns.

*Step 2.* $\operatorname{vec}(X) = (1, 0, 0, 1)^{\top}$ and $I_2 \otimes A$ is block diagonal with
two copies of $A$, so the product is $(A_{:,1}, A_{:,2}) = (1, 3, 2, 4)^{\top}$.

$$
\boxed{\operatorname{vec}(AX) = (I \otimes A)\operatorname{vec}(X) = (1, 3, 2, 4)^{\top}}
$$

**Key takeaway.** $I \otimes A$ applies $A$ to every column; $B^{\top} \otimes I$ mixes the
columns. Theorem 4.3 is the general statement.

In [10]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
X = np.eye(2)
print("vec(A X)             :", vec(A @ X))
print("(I kron A) vec(X)    :", np.kron(np.eye(2), A) @ vec(X))
assert np.allclose(vec(A @ X), np.kron(np.eye(2), A) @ vec(X))
assert np.allclose(vec(A @ X), [1.0, 3.0, 2.0, 4.0])

vec(A X)             : [1. 3. 2. 4.]
(I kron A) vec(X)    : [1. 3. 2. 4.]


### Problem L0.10 — One step of a column-stochastic chain

**Statement.** For $P = \left[\begin{smallmatrix}0.8&0.3\\0.2&0.7\end{smallmatrix}\right]$ and
$\pi_0 = (1,0)^{\top}$, compute $\pi_1 = P\pi_0$ and verify that it is a distribution.

**Intuition.** Column $j$ of $P$ is the distribution of where the chain goes from state $j$, so
starting in state $1$ just reads off column $1$.

**Solution.**

*Step 1.* $\pi_1 = P e_1 = (0.8, 0.2)^{\top}$, the first column.

*Step 2.* Its entries are non-negative and sum to $1$, which is exactly the column-stochastic
condition $\mathbf{1}^{\top}P = \mathbf{1}^{\top}$ of Definition 3.7.

$$
\boxed{\pi_1 = (0.8, 0.2)^{\top}, \qquad \mathbf{1}^{\top}\pi_1 = 1}
$$

**Key takeaway.** Column-stochasticity is precisely the statement that one step maps
distributions to distributions.

In [11]:
P = np.array([[0.8, 0.3], [0.2, 0.7]])
pi0 = np.array([1.0, 0.0])
pi1 = P @ pi0
print("column sums:", P.sum(axis=0), "  pi1 =", pi1, "  sum =", pi1.sum())
assert np.allclose(P.sum(axis=0), 1.0) and np.allclose(pi1, [0.8, 0.2])

column sums: [1. 1.]   pi1 = [0.8 0.2]   sum = 1.0


## L1 — Foundations

### Problem L1.1 — Gradient of a general quadratic form

**Statement.** For $A \in \mathbb{R}^{n \times n}$ not assumed symmetric, derive
$\nabla_x (x^{\top}Ax)$ and specialize to symmetric $A$.

**Intuition.** The quadratic form is bilinear in the two copies of $x$, so the product rule
produces one term for each.

**Solution.**

*Step 1.* Take the differential:

$$
d(x^{\top}Ax) = (dx)^{\top}Ax + x^{\top}A\,dx .
$$

*Step 2.* The first term is a scalar, so it equals its own transpose,
$(dx)^{\top}Ax = x^{\top}A^{\top}dx$. Hence
$d(x^{\top}Ax) = x^{\top}(A + A^{\top})dx$.

*Step 3.* Theorem 4.1 in $\mathbb{R}^n$ reads off the gradient, and $A = A^{\top}$ collapses the
sum.

$$
\boxed{\nabla_x (x^{\top}Ax) = (A + A^{\top})x, \qquad = 2Ax \text{ when } A^{\top} = A}
$$

**Key takeaway.** Only the symmetric part of $A$ is visible to the quadratic form, since
$x^{\top}Ax = x^{\top}\tfrac{A + A^{\top}}{2}x$ for every $x$.

In [12]:
A = np.array([[1.0, 2.0], [0.0, 3.0]])
x = np.array([0.7, -1.3])
h = 1e-6
fd = np.array([((x + h * np.eye(2)[i]) @ A @ (x + h * np.eye(2)[i])
                - (x - h * np.eye(2)[i]) @ A @ (x - h * np.eye(2)[i])) / (2 * h) for i in range(2)])
print("(A + A^T) x       :", (A + A.T) @ x)
print("finite difference :", fd)
print("2 A x (wrong here):", 2 * A @ x)
assert np.abs(fd - (A + A.T) @ x).max() < 1e-8

(A + A^T) x       : [-1.2 -6.4]
finite difference : [-1.2 -6.4]
2 A x (wrong here): [-3.8 -7.8]


### Problem L1.2 — Gradient and Hessian of the least-squares loss

**Statement.** For $J(w) = \tfrac12 \lVert Xw - y \rVert_2^2$ with $X \in \mathbb{R}^{N \times d}$,
compute $\nabla_w J$ and $\nabla_w^2 J$, and give the stationarity condition.

**Intuition.** The loss is a convex quadratic, so its gradient is affine and its Hessian is the
constant Gram matrix.

**Solution.**

*Step 1.* Expand:

$$
J(w) = \tfrac12 w^{\top}X^{\top}Xw - y^{\top}Xw + \tfrac12 y^{\top}y .
$$

*Step 2.* $X^{\top}X$ is symmetric, so Problem L1.1 gives
$\nabla_w \bigl( \tfrac12 w^{\top}X^{\top}Xw \bigr) = X^{\top}Xw$, and Problem L0.3 gives
$\nabla_w (y^{\top}Xw) = X^{\top}y$.

*Step 3.* Differentiating the affine gradient once more leaves the constant matrix
$X^{\top}X$.

$$
\boxed{\nabla_w J = X^{\top}(Xw - y), \qquad \nabla_w^2 J = X^{\top}X, \qquad X^{\top}Xw = X^{\top}y}
$$

**Key takeaway.** The normal equations are just $\nabla J = 0$, and $\nabla^2 J = X^{\top}X \succeq 0$
is why the stationary point is a global minimum.

In [13]:
Xd = rng.standard_normal((30, 4))
yd = rng.standard_normal(30)
w = rng.standard_normal(4)
Jf = lambda w_: 0.5 * np.sum((Xd @ w_ - yd) ** 2)
h = 1e-6
fd = np.array([(Jf(w + h * np.eye(4)[i]) - Jf(w - h * np.eye(4)[i])) / (2 * h) for i in range(4)])
print("analytic gradient :", Xd.T @ (Xd @ w - yd))
print("finite difference :", fd)
w_star = np.linalg.solve(Xd.T @ Xd, Xd.T @ yd)
print("normal-equation solution :", w_star)
print("lstsq solution           :", np.linalg.lstsq(Xd, yd, rcond=None)[0])
print("||grad at w_star||       :", np.linalg.norm(Xd.T @ (Xd @ w_star - yd)))
assert np.abs(fd - Xd.T @ (Xd @ w - yd)).max() < 1e-6
assert np.allclose(w_star, np.linalg.lstsq(Xd, yd, rcond=None)[0])

analytic gradient : [  3.8936   5.165  -36.2231  25.5867]
finite difference : [  3.8936   5.165  -36.2231  25.5867]
normal-equation solution : [ 0.2104 -0.098  -0.2122  0.0414]
lstsq solution           : [ 0.2104 -0.098  -0.2122  0.0414]
||grad at w_star||       : 7.043575467574348e-16


### Problem L1.3 — Derivative of a trace form

**Statement.** For $A \in \mathbb{R}^{p \times m}$, $X \in \mathbb{R}^{m \times n}$,
$B \in \mathbb{R}^{n \times p}$, derive $\nabla_X \operatorname{tr}(AXB)$.

**Intuition.** The function is linear in $X$, so the gradient is a constant matrix built from $A$
and $B$.

**Solution.**

*Step 1.* Linearity of the trace gives
$d\operatorname{tr}(AXB) = \operatorname{tr}(A(dX)B)$.

*Step 2.* Cyclicity moves $B$ around the front: $\operatorname{tr}(A(dX)B) = \operatorname{tr}(BA\,dX)$.

*Step 3.* Write $\operatorname{tr}(BA\,dX) = \operatorname{tr}\bigl( ((BA)^{\top})^{\top}dX \bigr)$
and apply Theorem 4.1.

$$
\boxed{\nabla_X \operatorname{tr}(AXB) = (BA)^{\top} = A^{\top}B^{\top} \in \mathbb{R}^{m \times n}}
$$

**Key takeaway.** Cyclic permutation is the one move that turns any trace expression into the
canonical form $\operatorname{tr}(G^{\top}dX)$.

In [14]:
p_, m_, n_ = 3, 4, 5
A = rng.standard_normal((p_, m_))
B = rng.standard_normal((n_, p_))
X = rng.standard_normal((m_, n_))
G = A.T @ B.T
h = 1e-6
fd = np.zeros_like(X)
for idx in np.ndindex(X.shape):
    E = np.zeros_like(X)
    E[idx] = h
    fd[idx] = (np.trace(A @ (X + E) @ B) - np.trace(A @ (X - E) @ B)) / (2 * h)
print("shape of gradient:", G.shape, " matches X:", G.shape == X.shape)
print("max |A^T B^T - finite difference| =", np.abs(fd - G).max())
assert np.abs(fd - G).max() < 1e-7

shape of gradient: (4, 5)  matches X: True
max |A^T B^T - finite difference| = 7.716156602555202e-10


### Problem L1.4 — Derivative of $\operatorname{tr}(X^{\top}AX)$

**Statement.** For symmetric $A \in \mathbb{R}^{m \times m}$ and $X \in \mathbb{R}^{m \times n}$,
derive $\nabla_X \operatorname{tr}(X^{\top}AX)$.

**Intuition.** This is the matrix version of $x^{\top}Ax$, so the answer should be the matrix
version of $2Ax$.

**Solution.**

*Step 1.* The product rule gives

$$
d\operatorname{tr}(X^{\top}AX) = \operatorname{tr}\bigl( (dX)^{\top}AX \bigr) + \operatorname{tr}\bigl( X^{\top}A\,dX \bigr) .
$$

*Step 2.* Using $\operatorname{tr}(M^{\top}) = \operatorname{tr}(M)$ on the first term,
$\operatorname{tr}((dX)^{\top}AX) = \operatorname{tr}(X^{\top}A^{\top}dX)$, which equals
$\operatorname{tr}(X^{\top}A\,dX)$ because $A$ is symmetric.

*Step 3.* The two terms coincide, so
$d\operatorname{tr}(X^{\top}AX) = \operatorname{tr}\bigl( (2AX)^{\top}dX \bigr)$.

$$
\boxed{\nabla_X \operatorname{tr}(X^{\top}AX) = 2AX}
$$

**Key takeaway.** Symmetry of $A$ is what merges the two terms; for general $A$ the answer is
$(A + A^{\top})X$, exactly as in Problem L1.1.

In [15]:
A = rng.standard_normal((4, 4))
A = A + A.T
X = rng.standard_normal((4, 3))
G = 2 * A @ X
h = 1e-6
fd = np.zeros_like(X)
for idx in np.ndindex(X.shape):
    E = np.zeros_like(X)
    E[idx] = h
    fd[idx] = (np.trace((X + E).T @ A @ (X + E)) - np.trace((X - E).T @ A @ (X - E))) / (2 * h)
print("max |2 A X - finite difference| =", np.abs(fd - G).max())
Ans = rng.standard_normal((4, 4))
fd2 = np.zeros_like(X)
for idx in np.ndindex(X.shape):
    E = np.zeros_like(X)
    E[idx] = h
    fd2[idx] = (np.trace((X + E).T @ Ans @ (X + E)) - np.trace((X - E).T @ Ans @ (X - E))) / (2 * h)
print("non-symmetric case: max |(A + A^T) X - fd| =", np.abs(fd2 - (Ans + Ans.T) @ X).max())
assert np.abs(fd - G).max() < 1e-6 and np.abs(fd2 - (Ans + Ans.T) @ X).max() < 1e-6

max |2 A X - finite difference| = 3.5741116732879163e-09
non-symmetric case: max |(A + A^T) X - fd| = 8.177123977937129e-10


### Problem L1.5 — Gradient of the log-determinant

**Statement.** For $X \in \mathbb{R}^{n \times n}$ with $\det X \gt 0$, derive
$\nabla_X \ln\det X$, and state what changes if $X$ is constrained to be symmetric.

**Intuition.** The determinant measures volume, and the relative change of a volume under a small
distortion is the trace of that distortion in the coordinates $X$ provides.

**Solution.**

*Step 1.* Jacobi's formula, proved as Proof 5.2 (3), gives
$d(\det X) = \det(X)\operatorname{tr}(X^{-1}dX)$.

*Step 2.* The scalar chain rule for $\ln$ divides by $\det X$:

$$
d(\ln\det X) = \operatorname{tr}(X^{-1}dX) = \operatorname{tr}\bigl( (X^{-\top})^{\top}dX \bigr) .
$$

*Step 3.* Theorem 4.1 identifies the gradient. If instead $X$ is parameterized by its upper
triangle, perturbing an off-diagonal free parameter moves two entries, so the derivative with
respect to that parameter doubles.

$$
\boxed{\nabla_X \ln\det X = X^{-\top}; \quad \text{under a symmetry constraint, } 2X^{-1} - \operatorname{diag}(X^{-1})}
$$

**Key takeaway.** This gradient is the engine of Gaussian maximum likelihood, and the symmetric
correction is the most common sign of trouble in a hand-written covariance optimizer.

In [16]:
M = rng.standard_normal((4, 4))
Xs = M @ M.T + 4 * np.eye(4)
G = np.linalg.inv(Xs).T
h = 1e-6
fd = np.zeros_like(Xs)
for idx in np.ndindex(Xs.shape):
    E = np.zeros_like(Xs)
    E[idx] = h
    fd[idx] = (np.log(np.linalg.det(Xs + E)) - np.log(np.linalg.det(Xs - E))) / (2 * h)
print("max |X^-T - finite difference| (unconstrained) =", np.abs(fd - G).max())
i, j = 0, 1
Esym = np.zeros_like(Xs)
Esym[i, j] = Esym[j, i] = h
fd_sym = (np.log(np.linalg.det(Xs + Esym)) - np.log(np.linalg.det(Xs - Esym))) / (2 * h)
print(f"symmetric-parameter derivative at ({i},{j}) = {fd_sym:.6f}"
      f"   2*(X^-1)_[{i},{j}] = {2 * np.linalg.inv(Xs)[i, j]:.6f}"
      f"   unconstrained entry = {G[i, j]:.6f}")
assert np.abs(fd - G).max() < 1e-7
assert abs(fd_sym - 2 * np.linalg.inv(Xs)[i, j]) < 1e-6

max |X^-T - finite difference| (unconstrained) = 9.273382339802794e-10
symmetric-parameter derivative at (0,1) = 0.030985   2*(X^-1)_[0,1] = 0.030985   unconstrained entry = 0.015493


### Problem L1.6 — Jacobi's formula

**Statement.** Prove $\dfrac{d}{dt}\det A(t) = \det A(t)\operatorname{tr}\bigl( A(t)^{-1}\dot{A}(t) \bigr)$
for a smooth invertible $A(t)$, and deduce $\nabla_A \det A = \det(A) A^{-\top} = \operatorname{adj}(A)^{\top}$.

**Intuition.** Factor the perturbed matrix as the original times a near-identity; the determinant
of a near-identity is $1 + \operatorname{tr}$ of the small part.

**Solution.**

*Step 1.* Write $A(t + \epsilon) = A(t)\bigl( I + \epsilon A(t)^{-1}\dot{A}(t) \bigr) + O(\epsilon^2)$
and use multiplicativity of the determinant.

*Step 2.* For small $M$, the eigenvalue expansion of Proof 5.2 (3) gives
$\det(I + \epsilon M) = 1 + \epsilon\operatorname{tr}(M) + O(\epsilon^2)$.

*Step 3.* Subtract $\det A(t)$, divide by $\epsilon$ and let $\epsilon \to 0$.

*Step 4.* Reading $d(\det A) = \operatorname{tr}\bigl( (\det(A)A^{-\top})^{\top}dA \bigr)$ through
Theorem 4.1 gives the gradient, and Cramer's rule $A^{-1} = \operatorname{adj}(A)/\det A$ rewrites it.

$$
\boxed{\frac{d}{dt}\det A(t) = \det A(t)\operatorname{tr}\bigl( A^{-1}\dot{A} \bigr), \qquad \nabla_A \det A = \operatorname{adj}(A)^{\top}}
$$

**Key takeaway.** Jacobi's formula is the volume version of the chain rule, and it is the single
ingredient behind $\nabla_X \ln\det X = X^{-\top}$.

In [17]:
A0 = rng.standard_normal((4, 4))
Ad = rng.standard_normal((4, 4))
At = lambda t: A0 + t * Ad
h = 1e-6
lhs = (np.linalg.det(At(h)) - np.linalg.det(At(-h))) / (2 * h)
rhs = np.linalg.det(A0) * np.trace(np.linalg.inv(A0) @ Ad)
print(f"d/dt det A(t) at 0 : {lhs:.10f}")
print(f"det(A) tr(A^-1 A') : {rhs:.10f}")
grad = np.linalg.det(A0) * np.linalg.inv(A0).T
fd = np.zeros_like(A0)
for idx in np.ndindex(A0.shape):
    E = np.zeros_like(A0)
    E[idx] = h
    fd[idx] = (np.linalg.det(A0 + E) - np.linalg.det(A0 - E)) / (2 * h)
print("max |det(A) A^-T - finite difference| =", np.abs(fd - grad).max())
assert abs(lhs - rhs) < 1e-5 and np.abs(fd - grad).max() < 1e-5

d/dt det A(t) at 0 : -2.5302835016
det(A) tr(A^-1 A') : -2.5302835017
max |det(A) A^-T - finite difference| = 5.005313941097711e-10


### Problem L1.7 — Derivative of a time-varying inverse

**Statement.** Let $A(t)$ be smooth and invertible. Derive $\dfrac{d}{dt}\bigl( A(t)^{-1} \bigr)$.

**Intuition.** Differentiating $AA^{-1} = I$ and solving for the unknown derivative is the matrix
version of the quotient rule.

**Solution.**

*Step 1.* Differentiate $A(t)A(t)^{-1} = I$ with the product rule:

$$
\dot{A}A^{-1} + A\frac{d}{dt}\bigl( A^{-1} \bigr) = 0 .
$$

*Step 2.* Left-multiply by $A^{-1}$.

$$
\boxed{\frac{d}{dt}\bigl( A(t)^{-1} \bigr) = -A(t)^{-1}\dot{A}(t)A(t)^{-1}}
$$

**Key takeaway.** The two sandwiching inverses are not optional: matrices do not commute, so the
scalar form $-\dot{a}/a^2$ has to be written with the order kept.

In [18]:
A0 = rng.standard_normal((4, 4)) + 4 * np.eye(4)
Ad = rng.standard_normal((4, 4))
h = 1e-6
lhs = (np.linalg.inv(A0 + h * Ad) - np.linalg.inv(A0 - h * Ad)) / (2 * h)
rhs = -np.linalg.inv(A0) @ Ad @ np.linalg.inv(A0)
print("max |finite difference - (-A^-1 A' A^-1)| =", np.abs(lhs - rhs).max())
wrong = -np.linalg.inv(A0) @ np.linalg.inv(A0) @ Ad
print("max |finite difference - (-A^-2 A')|      =", np.abs(lhs - wrong).max(), " (order matters)")
assert np.abs(lhs - rhs).max() < 1e-6

max |finite difference - (-A^-1 A' A^-1)| = 4.391965307062873e-11
max |finite difference - (-A^-2 A')|      = 0.0927171597668704  (order matters)


### Problem L1.8 — The Jacobian of $X \mapsto AXB$

**Statement.** Compute $\dfrac{\partial \operatorname{vec}(AXB)}{\partial \operatorname{vec}(X)}$.

**Intuition.** The map is linear, so its Jacobian is the matrix of the map itself, written in the
$\operatorname{vec}$ coordinates.

**Solution.**

*Step 1.* Linearity gives $d(AXB) = A(dX)B$.

*Step 2.* Apply Theorem 4.3 to the differential:
$\operatorname{vec}(A(dX)B) = (B^{\top}\otimes A)\operatorname{vec}(dX)$.

*Step 3.* Matching against $\operatorname{vec}(dY) = J\operatorname{vec}(dX)$ identifies $J$.

$$
\boxed{\frac{\partial \operatorname{vec}(AXB)}{\partial \operatorname{vec}(X)} = B^{\top}\otimes A}
$$

**Key takeaway.** Every linear matrix equation becomes an ordinary linear system in the
$\operatorname{vec}$ coordinates, at the cost of squaring the dimension.

In [19]:
A = rng.standard_normal((3, 4))
B = rng.standard_normal((5, 2))
X = rng.standard_normal((4, 5))
J = np.kron(B.T, A)
h = 1e-6
Jfd = np.zeros((3 * 2, 4 * 5))
for k in range(4 * 5):
    E = np.zeros(4 * 5)
    E[k] = h
    Ek = E.reshape(4, 5, order="F")
    Jfd[:, k] = (vec(A @ (X + Ek) @ B) - vec(A @ (X - Ek) @ B)) / (2 * h)
print("Jacobian shape:", J.shape, " max |J - finite difference| =", np.abs(J - Jfd).max())
assert np.abs(J - Jfd).max() < 1e-8

Jacobian shape: (6, 20)  max |J - finite difference| = 1.365354052040857e-09


### Problem L1.9 — The Laplacian quadratic form

**Statement.** For a weighted undirected graph with $A_{ij} \ge 0$, express $x^{\top}Lx$ as a sum
over edges and deduce $L \succeq 0$.

**Intuition.** The Laplacian measures how much a signal disagrees with itself across edges.

**Solution.**

*Step 1.* Expand $x^{\top}Lx = \sum_i d_i x_i^2 - \sum_{i,j}A_{ij}x_ix_j$.

*Step 2.* Substitute $d_i = \sum_j A_{ij}$ and split the first sum symmetrically:

$$
\sum_i d_i x_i^2 = \tfrac12\sum_{i,j}A_{ij}x_i^2 + \tfrac12\sum_{i,j}A_{ij}x_j^2 ,
$$

using $A_{ij} = A_{ji}$.

*Step 3.* Combine into a perfect square:

$$
x^{\top}Lx = \tfrac12\sum_{i,j}A_{ij}\bigl( x_i^2 - 2x_ix_j + x_j^2 \bigr) = \sum_{\lbrace i,j \rbrace \in E}A_{ij}(x_i - x_j)^2 .
$$

*Step 4.* Every summand is a non-negative weight times a square.

$$
\boxed{x^{\top}Lx = \sum_{\lbrace i,j \rbrace \in E}A_{ij}(x_i - x_j)^2 \ \ge 0 \implies L \succeq 0}
$$

**Key takeaway.** Non-negativity of the weights is the whole hypothesis; a single negative weight
makes $L$ indefinite, as the code cell shows.

In [20]:
Aw = np.array([[0.0, 2.0, 0.0], [2.0, 0.0, 0.5], [0.0, 0.5, 0.0]])
Lw = np.diag(Aw.sum(1)) - Aw
xs = rng.standard_normal(3)
edge_sum = sum(Aw[i, j] * (xs[i] - xs[j]) ** 2 for i in range(3) for j in range(i + 1, 3))
print("x^T L x   =", xs @ Lw @ xs, "   edge sum =", edge_sum)
print("spectrum of L:", np.linalg.eigvalsh(Lw))
Aneg = Aw.copy()
Aneg[0, 1] = Aneg[1, 0] = -2.0
Lneg = np.diag(Aneg.sum(1)) - Aneg
print("with one negative weight, spectrum:", np.linalg.eigvalsh(Lneg), " -> indefinite")
assert abs(xs @ Lw @ xs - edge_sum) < 1e-12
assert np.linalg.eigvalsh(Lw).min() > -1e-12 and np.linalg.eigvalsh(Lneg).min() < -1e-6

x^T L x   = 0.21910558401355876    edge sum = 0.21910558401355873
spectrum of L: [-0.      0.6972  4.3028]
with one negative weight, spectrum: [-3.7913 -0.      0.7913]  -> indefinite


### Problem L1.10 — The incidence factorization $L = BB^{\top}$

**Statement.** Let $B \in \mathbb{R}^{n \times m}$ be an oriented incidence matrix of an unweighted
graph. Prove $BB^{\top} = L$.

**Intuition.** $B^{\top}$ takes a vertex signal to the vector of its differences along edges, and
$B$ sums those differences back at the vertices.

**Solution.**

*Step 1.* For an edge $e$ oriented $u \to v$, the column is $B_{:,e} = e_u - e_v$, so

$$
(BB^{\top})_{ij} = \sum_{e \in E} B_{ie}B_{je} .
$$

*Step 2.* For $i = j$ every incident edge contributes $(\pm 1)^2 = 1$, giving $d_i$.

*Step 3.* For $i \neq j$ only the edge $\lbrace i,j \rbrace$ contributes, and its two entries have
opposite signs, giving $-A_{ij}$.

*Step 4.* Combining, $(BB^{\top})_{ij} = D_{ij} - A_{ij}$, and the orientation cancels because it
enters squared.

$$
\boxed{BB^{\top} = D - A = L, \qquad x^{\top}Lx = \lVert B^{\top}x \rVert_2^2}
$$

**Key takeaway.** The factorization proves $L \succeq 0$ in one line and identifies
$\operatorname{Null}(L) = \operatorname{Null}(B^{\top})$, which is Proof 5.5 Step 3.

In [21]:
Ai = np.array([[0.0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]])
Li = np.diag(Ai.sum(1)) - Ai
ed = [(i, j) for i in range(4) for j in range(i + 1, 4) if Ai[i, j] > 0]
Bi = np.zeros((4, len(ed)))
for k, (i, j) in enumerate(ed):
    Bi[i, k], Bi[j, k] = 1.0, -1.0
xs = rng.standard_normal(4)
print("||L - B B^T||_F =", np.linalg.norm(Li - Bi @ Bi.T))
print("x^T L x =", xs @ Li @ xs, "  ||B^T x||^2 =", np.linalg.norm(Bi.T @ xs) ** 2)
Bflip = Bi.copy()
Bflip[:, 0] *= -1
print("after flipping one orientation, ||L - B B^T||_F =", np.linalg.norm(Li - Bflip @ Bflip.T))
assert np.linalg.norm(Li - Bi @ Bi.T) < 1e-12
assert np.linalg.norm(Li - Bflip @ Bflip.T) < 1e-12

||L - B B^T||_F = 0.0
x^T L x = 10.824737491246147   ||B^T x||^2 = 10.824737491246147
after flipping one orientation, ||L - B B^T||_F = 0.0


### Problem L1.11 — Quadratic form of the normalized Laplacian

**Statement.** For a graph with all $d_i \gt 0$, derive $x^{\top}L_{\mathrm{sym}}x$ as a sum over
edges.

**Intuition.** $L_{\mathrm{sym}}$ is $L$ read in coordinates rescaled by $\sqrt{d_i}$, so the same
edge sum appears with rescaled signal values.

**Solution.**

*Step 1.* By definition,
$x^{\top}L_{\mathrm{sym}}x = (D^{-1/2}x)^{\top}L(D^{-1/2}x)$.

*Step 2.* Put $y = D^{-1/2}x$, so $y_i = x_i/\sqrt{d_i}$, and apply Problem L1.9 to $y$.

$$
\boxed{x^{\top}L_{\mathrm{sym}}x = \sum_{\lbrace i,j \rbrace \in E}A_{ij}\left( \frac{x_i}{\sqrt{d_i}} - \frac{x_j}{\sqrt{d_j}} \right)^{2}}
$$

**Key takeaway.** Normalizing puts the spectrum in $[0,2]$ by Proof 5.5 Step 4, which is what
keeps deep graph networks from exploding.

In [22]:
Ac = np.array([[0.0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 3], [0, 0, 3, 0]])
dc = Ac.sum(1)
Lc = np.diag(dc) - Ac
Ls = np.diag(dc ** -0.5) @ Lc @ np.diag(dc ** -0.5)
xs = rng.standard_normal(4)
edge_sum = sum(Ac[i, j] * (xs[i] / np.sqrt(dc[i]) - xs[j] / np.sqrt(dc[j])) ** 2
               for i in range(4) for j in range(i + 1, 4))
print("x^T L_sym x =", xs @ Ls @ xs, "   edge sum =", edge_sum)
print("spectrum of L_sym:", np.linalg.eigvalsh(Ls), " -> contained in [0, 2]")
assert abs(xs @ Ls @ xs - edge_sum) < 1e-12
assert np.linalg.eigvalsh(Ls).max() <= 2 + 1e-12

x^T L_sym x = 1.147708399884789    edge sum = 1.147708399884789
spectrum of L_sym: [-0.      0.6479  1.5     1.8521]  -> contained in [0, 2]


### Problem L1.12 — $L_{\mathrm{rw}}$ and $L_{\mathrm{sym}}$ are similar

**Statement.** Show that $L_{\mathrm{rw}} = D^{-1}L$ and $L_{\mathrm{sym}} = D^{-1/2}LD^{-1/2}$
have the same eigenvalues, and relate their eigenvectors.

**Intuition.** They are the same operator written in two coordinate systems that differ by the
diagonal rescaling $D^{1/2}$.

**Solution.**

*Step 1.* Insert $D^{-1/2}D^{1/2} = I$:

$$
L_{\mathrm{rw}} = D^{-1}L = D^{-1/2}\bigl( D^{-1/2}LD^{-1/2} \bigr)D^{1/2} = D^{-1/2}L_{\mathrm{sym}}D^{1/2} .
$$

*Step 2.* This is a similarity with $S = D^{-1/2}$, and similar matrices share a characteristic
polynomial: $\det(S L_{\mathrm{sym}} S^{-1} - \lambda I) = \det(L_{\mathrm{sym}} - \lambda I)$.

*Step 3.* If $L_{\mathrm{sym}}v = \lambda v$ then $u = D^{-1/2}v$ satisfies
$L_{\mathrm{rw}}u = \lambda u$.

$$
\boxed{L_{\mathrm{rw}} = D^{-1/2}L_{\mathrm{sym}}D^{1/2} \implies \operatorname{spec}(L_{\mathrm{rw}}) = \operatorname{spec}(L_{\mathrm{sym}})}
$$

**Key takeaway.** Use $L_{\mathrm{sym}}$ for numerics — it is symmetric, so `eigh` applies — and
$L_{\mathrm{rw}}$ for probabilistic interpretation.

In [23]:
Lrw = np.diag(1.0 / dc) @ Lc
print("spec(L_rw) :", np.sort(np.linalg.eigvals(Lrw).real))
print("spec(L_sym):", np.linalg.eigvalsh(Ls))
w, V = np.linalg.eigh(Ls)
u = np.diag(dc ** -0.5) @ V[:, 1]
print("||L_rw u - lambda u|| =", np.linalg.norm(Lrw @ u - w[1] * u))
assert np.allclose(np.sort(np.linalg.eigvals(Lrw).real), np.linalg.eigvalsh(Ls))
assert np.linalg.norm(Lrw @ u - w[1] * u) < 1e-12

spec(L_rw) : [0.     0.6479 1.5    1.8521]
spec(L_sym): [-0.      0.6479  1.5     1.8521]
||L_rw u - lambda u|| = 2.2929868617541516e-16


### Problem L1.13 — Algebraic connectivity vanishes exactly when the graph disconnects

**Statement.** Prove $\lambda_2(L) = 0$ if and only if $G$ is disconnected.

**Intuition.** The null space of $L$ is the space of signals constant on each component, so it is
larger than one dimension exactly when there is more than one component.

**Solution.**

*Step 1.* By Proof 5.5 Step 3, $\operatorname{Null}(L)$ is spanned by the indicator vectors of the
connected components, so its dimension is the number $c$ of components.

*Step 2.* ($\Leftarrow$) If $G$ is disconnected then $c \ge 2$, so $L$ has at least two
independent null vectors, and with the ascending ordering $\lambda_1 = \lambda_2 = 0$.

*Step 3.* ($\Rightarrow$) If $\lambda_2(L) = 0$ then $\dim\operatorname{Null}(L) \ge 2$, so
$c \ge 2$ and $G$ is disconnected.

$$
\boxed{\lambda_2(L) = 0 \iff G \text{ is disconnected}, \qquad \dim\operatorname{Null}(L) = \#\text{components}}
$$

**Key takeaway.** $\lambda_2$ is a continuous measure of how nearly disconnected a graph is, which
is exactly what Theorem 4.6 quantifies.

In [24]:
Aconn = np.array([[0.0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]])
Adisc = Aconn.copy()
Adisc[2, 3] = Adisc[3, 2] = 0.0
for name, Am in (("connected", Aconn), ("disconnected", Adisc)):
    Lm = np.diag(Am.sum(1)) - Am
    lam = np.linalg.eigvalsh(Lm)
    comps = Lm.shape[0] - np.linalg.matrix_rank(Lm)
    print(f"{name:14s} spectrum {np.round(lam, 6)}   lambda_2 = {lam[1]:.6f}   components = {comps}")
assert np.linalg.eigvalsh(np.diag(Aconn.sum(1)) - Aconn)[1] > 1e-8
assert abs(np.linalg.eigvalsh(np.diag(Adisc.sum(1)) - Adisc)[1]) < 1e-12

connected      spectrum [-0.  1.  3.  4.]   lambda_2 = 1.000000   components = 1
disconnected   spectrum [-0.  0.  3.  3.]   lambda_2 = 0.000000   components = 2


### Problem L1.14 — $\lambda = 1$ is an eigenvalue of every column-stochastic matrix

**Statement.** Let $P$ be column-stochastic. Show that $1 \in \operatorname{spec}(P)$ and that
$\rho(P) = 1$.

**Intuition.** Total probability is conserved, and conservation is an eigenvalue equation for the
all-ones vector on the transposed side.

**Solution.**

*Step 1.* Column sums equal one means $\mathbf{1}^{\top}P = \mathbf{1}^{\top}$, that is
$P^{\top}\mathbf{1} = \mathbf{1}$.

*Step 2.* $P$ and $P^{\top}$ have the same characteristic polynomial, because
$\det(M^{\top}) = \det(M)$. Hence $1 \in \operatorname{spec}(P)$ and $\rho(P) \ge 1$.

*Step 3.* For the reverse bound, $\lVert P \rVert_1 = \max_j\sum_i \lvert P_{ij} \rvert = 1$, and
the spectral radius never exceeds an induced norm: if $Pv = \lambda v$ with $v \neq 0$ then
$\lvert \lambda \rvert \lVert v \rVert_1 = \lVert Pv \rVert_1 \le \lVert P \rVert_1 \lVert v \rVert_1$.

$$
\boxed{P^{\top}\mathbf{1} = \mathbf{1} \implies 1 \in \operatorname{spec}(P), \qquad \rho(P) = \lVert P \rVert_1 = 1}
$$

**Key takeaway.** Step 3 is the half most often skipped: $\mathbf{1}^{\top}P = \mathbf{1}^{\top}$
alone gives only $\rho(P) \ge 1$, and the norm bound is what closes it.

In [25]:
Pc = np.array([[0.5, 0.2, 0.1], [0.3, 0.6, 0.3], [0.2, 0.2, 0.6]])
print("column sums:", Pc.sum(axis=0))
ev = np.linalg.eigvals(Pc)
print("eigenvalues:", ev)
print("spectral radius:", np.abs(ev).max(), "   ||P||_1 =", np.linalg.norm(Pc, 1))
assert np.allclose(Pc.sum(axis=0), 1.0)
assert abs(np.abs(ev).max() - 1.0) < 1e-12
assert abs(np.linalg.norm(Pc, 1) - 1.0) < 1e-12

column sums: [1. 1. 1.]
eigenvalues: [1.  0.3 0.4]
spectral radius: 1.0    ||P||_1 = 1.0


### Problem L1.15 — Matrix exponential of a $2 \times 2$ Jordan block

**Statement.** Compute $e^{Jt}$ for $J = \left[\begin{smallmatrix}\lambda & 1 \\ 0 & \lambda\end{smallmatrix}\right]$.

**Intuition.** Split the block into a scalar part and a nilpotent part; they commute, so the
exponentials multiply.

**Solution.**

*Step 1.* Write $J = \lambda I + N$ with $N = E_{12}$, and note $N^2 = 0$ and
$(\lambda I)N = N(\lambda I)$.

*Step 2.* Theorem 4.7 applies to the commuting pair:
$e^{Jt} = e^{\lambda t I}e^{Nt}$.

*Step 3.* $e^{\lambda t I} = e^{\lambda t}I$, and by Problem L0.5
$e^{Nt} = I + Nt = \left[\begin{smallmatrix}1 & t \\ 0 & 1\end{smallmatrix}\right]$.

$$
\boxed{e^{Jt} = e^{\lambda t}\begin{pmatrix}1 & t \\ 0 & 1\end{pmatrix}}
$$

**Key takeaway.** A defective eigenvalue produces a polynomial factor $t^{k}e^{\lambda t}$, which
is invisible in the spectrum and is exactly why an eigenbasis argument is not always available.

In [26]:
lam, t = -0.4, 1.3
Jb = np.array([[lam, 1.0], [0.0, lam]])
print("expm(J t):\n", sla.expm(Jb * t))
print("closed form:\n", np.exp(lam * t) * np.array([[1.0, t], [0.0, 1.0]]))
print("eigenvalues of J:", np.linalg.eigvals(Jb),
      "  rank of (J - lambda I):", np.linalg.matrix_rank(Jb - lam * np.eye(2)))
assert np.allclose(sla.expm(Jb * t), np.exp(lam * t) * np.array([[1.0, t], [0.0, 1.0]]))

expm(J t):
 [[0.5945 0.7729]
 [0.     0.5945]]
closed form:
 [[0.5945 0.7729]
 [0.     0.5945]]
eigenvalues of J: [-0.4 -0.4]   rank of (J - lambda I): 1


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Jacobian of the softmax

**Statement.** For $s(z)_i = e^{z_i} / \sum_k e^{z_k}$, derive the Jacobian $J_s(z)$.

**Intuition.** Raising one logit raises its own probability and lowers every other, because the
denominator is shared.

**Solution.**

*Step 1.* Write $s_i = u/v$ with $u = e^{z_i}$ and $v = \sum_k e^{z_k}$, so
$\partial v/\partial z_j = e^{z_j}$.

*Step 2.* For $i = j$ the quotient rule gives

$$
\frac{\partial s_i}{\partial z_i} = \frac{e^{z_i}v - e^{z_i}e^{z_i}}{v^2} = s_i(1 - s_i) .
$$

*Step 3.* For $i \neq j$ only the denominator depends on $z_j$:
$\partial s_i/\partial z_j = -s_is_j$.

*Step 4.* Assemble with the Kronecker delta.

$$
\boxed{J_s(z) = \operatorname{diag}(s) - ss^{\top}}
$$

**Key takeaway.** $J_s$ is symmetric positive semidefinite with $J_s\mathbf{1} = 0$: shifting every
logit by the same constant leaves the softmax unchanged.

In [27]:
z = rng.standard_normal(5)
s = np.exp(z - z.max())
s = s / s.sum()
Js = np.diag(s) - np.outer(s, s)
h = 1e-6
Jfd = np.zeros((5, 5))
for j in range(5):
    zp, zm = z.copy(), z.copy()
    zp[j] += h
    zm[j] -= h
    sp = np.exp(zp - zp.max()); sp /= sp.sum()
    sm = np.exp(zm - zm.max()); sm /= sm.sum()
    Jfd[:, j] = (sp - sm) / (2 * h)
print("max |analytic - finite difference| =", np.abs(Js - Jfd).max())
print("J s 1 =", Js @ np.ones(5), "   eigenvalues:", np.linalg.eigvalsh(Js))
assert np.abs(Js - Jfd).max() < 1e-8
assert np.allclose(Js @ np.ones(5), 0.0) and np.linalg.eigvalsh(Js).min() > -1e-12

max |analytic - finite difference| = 1.829786322460336e-11
J s 1 = [-0.  0. -0.  0.  0.]    eigenvalues: [-0.      0.0212  0.0252  0.0537  0.3421]


### Problem L2.2 — Softmax with cross-entropy

**Statement.** For $J(z) = -\sum_k y_k \ln p_k$ with $p = \operatorname{softmax}(z)$ and a
probability vector $y$, derive $\nabla_z J$.

**Intuition.** The logarithm in the loss cancels the exponential in the softmax, leaving a plain
residual.

**Solution.**

*Step 1.* By Problem L2.1, $\partial p_k/\partial z_i = p_k(\delta_{ki} - p_i)$.

*Step 2.* Chain rule:

$$
\frac{\partial J}{\partial z_i} = -\sum_k \frac{y_k}{p_k}\,p_k(\delta_{ki} - p_i) = -\sum_k y_k(\delta_{ki} - p_i) .
$$

*Step 3.* Split the sum: $\sum_k y_k\delta_{ki} = y_i$ and $\sum_k y_k = 1$, so the expression is
$-y_i + p_i$.

$$
\boxed{\nabla_z J = p - y}
$$

**Key takeaway.** The composite gradient is a residual with no Jacobian left in it, which is why
libraries fuse softmax and cross-entropy into one numerically stable operation.

In [28]:
z = rng.standard_normal(4)
y = np.zeros(4)
y[2] = 1.0
softmax = lambda v: np.exp(v - v.max()) / np.exp(v - v.max()).sum()
lossf = lambda v: -np.sum(y * np.log(softmax(v)))
h = 1e-6
fd = np.array([(lossf(z + h * np.eye(4)[i]) - lossf(z - h * np.eye(4)[i])) / (2 * h) for i in range(4)])
print("p - y             :", softmax(z) - y)
print("finite difference :", fd)
assert np.abs(fd - (softmax(z) - y)).max() < 1e-7

p - y             : [ 0.5209  0.1122 -0.8069  0.1738]
finite difference : [ 0.5209  0.1122 -0.8069  0.1738]


### Problem L2.3 — Gradients of a low-rank matrix factorization

**Statement.** For $f(U, V) = \tfrac12\lVert X - UV^{\top} \rVert_F^2$ with
$U \in \mathbb{R}^{m \times r}$, $V \in \mathbb{R}^{n \times r}$, compute $\nabla_U f$ and
$\nabla_V f$.

**Intuition.** Each factor sees the residual projected onto the other factor.

**Solution.**

*Step 1.* Write $R = UV^{\top} - X$ so that $f = \tfrac12\lVert R \rVert_F^2$ and
$df = \operatorname{tr}(R^{\top}dR)$.

*Step 2.* With $V$ fixed, $dR = (dU)V^{\top}$, so

$$
df = \operatorname{tr}\bigl( R^{\top}(dU)V^{\top} \bigr) = \operatorname{tr}\bigl( V^{\top}R^{\top}dU \bigr)
= \operatorname{tr}\bigl( (RV)^{\top}dU \bigr) .
$$

*Step 3.* With $U$ fixed, $dR = U(dV)^{\top}$ and the same manipulation gives
$df = \operatorname{tr}\bigl( (R^{\top}U)^{\top}dV \bigr)$.

$$
\boxed{\nabla_U f = (UV^{\top} - X)V, \qquad \nabla_V f = (UV^{\top} - X)^{\top}U}
$$

**Key takeaway.** Setting either gradient to zero with the other fixed gives the alternating
least-squares updates $U = XV(V^{\top}V)^{-1}$ and $V = X^{\top}U(U^{\top}U)^{-1}$.

In [29]:
mm, nn, rr = 6, 5, 2
Xm = rng.standard_normal((mm, nn))
U = rng.standard_normal((mm, rr))
V = rng.standard_normal((nn, rr))
fU = lambda U_: 0.5 * np.linalg.norm(Xm - U_ @ V.T) ** 2
gU = (U @ V.T - Xm) @ V
gV = (U @ V.T - Xm).T @ U
h = 1e-6
fdU = np.zeros_like(U)
for idx in np.ndindex(U.shape):
    E = np.zeros_like(U)
    E[idx] = h
    fdU[idx] = (fU(U + E) - fU(U - E)) / (2 * h)
print("max |grad_U - finite difference| =", np.abs(fdU - gU).max())
U_als = Xm @ V @ np.linalg.inv(V.T @ V)
print("ALS update residual ||grad_U(U_als)|| =", np.linalg.norm((U_als @ V.T - Xm) @ V))
assert np.abs(fdU - gU).max() < 1e-6
assert np.linalg.norm((U_als @ V.T - Xm) @ V) < 1e-10

max |grad_U - finite difference| = 9.041785986596551e-09
ALS update residual ||grad_U(U_als)|| = 1.366061580149424e-15


### Problem L2.4 — LoRA gradients and the balance identity

**Statement.** With $W = W_0 + BA$, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$
and $G = \nabla_W J$, derive $\nabla_B J$ and $\nabla_A J$, and prove
$\operatorname{tr}(B^{\top}\nabla_B J) = \operatorname{tr}(A\,\nabla_A J^{\top})$.

**Intuition.** Rescaling $B \to cB$, $A \to c^{-1}A$ leaves $BA$ untouched, so the two gradients
must carry the same amount of "radial" information.

**Solution.**

*Step 1.* $W_0$ is frozen, so $dW = (dB)A + B(dA)$ and
$dJ = \operatorname{tr}(G^{\top}dW)$ splits in two.

*Step 2.* Cyclicity on the first piece:
$\operatorname{tr}(G^{\top}(dB)A) = \operatorname{tr}(AG^{\top}dB) = \operatorname{tr}((GA^{\top})^{\top}dB)$.

*Step 3.* The second piece is already in canonical form:
$\operatorname{tr}(G^{\top}B\,dA) = \operatorname{tr}((B^{\top}G)^{\top}dA)$.

*Step 4.* Substitute the two gradients and use cyclicity once on each side:

$$
\operatorname{tr}\bigl( B^{\top}\nabla_B J \bigr) = \operatorname{tr}\bigl( B^{\top}GA^{\top} \bigr),
\qquad
\operatorname{tr}\bigl( \nabla_A J \, A^{\top} \bigr) = \operatorname{tr}\bigl( B^{\top}GA^{\top} \bigr) .
$$

The two scalars are literally the same expression, which proves the identity.

$$
\boxed{\nabla_B J = GA^{\top}, \qquad \nabla_A J = B^{\top}G, \qquad \operatorname{tr}(B^{\top}\nabla_B J) = \operatorname{tr}(\nabla_A J \, A^{\top})}
$$

**Key takeaway.** The invariance $BA = (cB)(c^{-1}A)$ makes the loss constant along a scaling
curve, and the balance identity is the derivative of that constancy.

In [30]:
d_, k_, r_ = 5, 4, 2
W0 = rng.standard_normal((d_, k_))
Bl = rng.standard_normal((d_, r_))
Al = rng.standard_normal((r_, k_))
Tgt = rng.standard_normal((d_, k_))
Jl = lambda B_, A_: 0.5 * np.linalg.norm(W0 + B_ @ A_ - Tgt) ** 2
G = W0 + Bl @ Al - Tgt
gB, gA = G @ Al.T, Bl.T @ G
h = 1e-6
fdB = np.zeros_like(Bl)
for idx in np.ndindex(Bl.shape):
    E = np.zeros_like(Bl)
    E[idx] = h
    fdB[idx] = (Jl(Bl + E, Al) - Jl(Bl - E, Al)) / (2 * h)
print("max |grad_B - finite difference| =", np.abs(fdB - gB).max())
print("tr(B^T grad_B) =", np.trace(Bl.T @ gB), "   tr(grad_A A^T) =", np.trace(gA @ Al.T))
c = 2.5
print("loss at (B, A) and at (cB, A/c):", Jl(Bl, Al), Jl(c * Bl, Al / c))
assert np.abs(fdB - gB).max() < 1e-6
assert abs(np.trace(Bl.T @ gB) - np.trace(gA @ Al.T)) < 1e-10
assert abs(Jl(Bl, Al) - Jl(c * Bl, Al / c)) < 1e-10

max |grad_B - finite difference| = 9.525792599163196e-09
tr(B^T grad_B) = 29.707262795669596    tr(grad_A A^T) = 29.707262795669592
loss at (B, A) and at (cB, A/c): 58.56083698022904 58.56083698022904


### Problem L2.5 — Backpropagation through a graph convolution

**Statement.** For a GCN layer $H = \sigma(\hat{A}XW)$ with $\hat{A}$ symmetric and fixed, and
upstream gradient $\Delta = \nabla_H J$, derive $\nabla_W J$ and $\nabla_X J$.

**Intuition.** The layer is a dense layer with one extra fixed matrix in front, so the chain rule
just carries $\hat{A}$ along.

**Solution.**

*Step 1.* Put $Z = \hat{A}XW$, so $H = \sigma(Z)$ elementwise and
$\nabla_Z J = \Delta \odot \sigma'(Z)$.

*Step 2.* With $X$ fixed, $dZ = \hat{A}X\,dW$ and

$$
dJ = \operatorname{tr}\bigl( (\nabla_Z J)^{\top}\hat{A}X\,dW \bigr)
= \operatorname{tr}\Bigl( \bigl( (\hat{A}X)^{\top}\nabla_Z J \bigr)^{\top}dW \Bigr) .
$$

*Step 3.* With $W$ fixed, $dZ = \hat{A}(dX)W$ and cyclicity gives
$dJ = \operatorname{tr}\bigl( (\hat{A}^{\top}(\nabla_Z J)W^{\top})^{\top}dX \bigr)$.

$$
\boxed{\nabla_W J = (\hat{A}X)^{\top}\bigl( \Delta \odot \sigma'(Z) \bigr), \qquad \nabla_X J = \hat{A}\bigl( \Delta \odot \sigma'(Z) \bigr)W^{\top}}
$$

**Key takeaway.** The graph enters the backward pass exactly as it entered the forward pass, so a
GCN costs the same asymptotically to train as to evaluate.

In [31]:
ng, din, dout = 7, 3, 4
Ag2 = (rng.random((ng, ng)) < 0.4).astype(float)
Ag2 = np.triu(Ag2, 1)
Ag2 = Ag2 + Ag2.T
At2 = Ag2 + np.eye(ng)
dt2 = At2.sum(1)
Ah = np.diag(dt2 ** -0.5) @ At2 @ np.diag(dt2 ** -0.5)
Xg = rng.standard_normal((ng, din))
Wg = rng.standard_normal((din, dout))
Tg = rng.standard_normal((ng, dout))
Jg = lambda W_, X_: 0.5 * np.linalg.norm(np.tanh(Ah @ X_ @ W_) - Tg) ** 2
Z = Ah @ Xg @ Wg
H = np.tanh(Z)
Delta = H - Tg
gZ = Delta * (1 - H ** 2)
gW = (Ah @ Xg).T @ gZ
gX = Ah @ gZ @ Wg.T
h = 1e-6
fdW = np.zeros_like(Wg)
for idx in np.ndindex(Wg.shape):
    E = np.zeros_like(Wg)
    E[idx] = h
    fdW[idx] = (Jg(Wg + E, Xg) - Jg(Wg - E, Xg)) / (2 * h)
fdX = np.zeros_like(Xg)
for idx in np.ndindex(Xg.shape):
    E = np.zeros_like(Xg)
    E[idx] = h
    fdX[idx] = (Jg(Wg, Xg + E) - Jg(Wg, Xg - E)) / (2 * h)
print("max |grad_W - finite difference| =", np.abs(fdW - gW).max())
print("max |grad_X - finite difference| =", np.abs(fdX - gX).max())
assert np.abs(fdW - gW).max() < 1e-6 and np.abs(fdX - gX).max() < 1e-6

max |grad_W - finite difference| = 4.530030017768638e-09
max |grad_X - finite difference| = 5.174298323140647e-09


### Problem L2.6 — Graph Fourier transform and spectral filtering

**Statement.** With $L = U\Lambda U^{\top}$, define $\hat{x} = U^{\top}x$ and show that applying a
scalar filter $g$ to each frequency is the same as computing $g(L)x$.

**Intuition.** The Laplacian eigenvectors are the graph's Fourier basis, and a filter is a
diagonal operator in that basis.

**Solution.**

*Step 1.* Transform: $\hat{x} = U^{\top}x$.

*Step 2.* Filter: $\hat{y} = g(\Lambda)\hat{x}$ with
$g(\Lambda) = \operatorname{diag}(g(\lambda_1), \dots, g(\lambda_n))$.

*Step 3.* Transform back: $y = U\hat{y} = Ug(\Lambda)U^{\top}x$, which is the definition of the
matrix function $g(L)$ for symmetric $L$.

$$
\boxed{y = Ug(\Lambda)U^{\top}x = g(L)x}
$$

**Key takeaway.** If $g$ is a degree-$K$ polynomial then $g(L)$ is a polynomial in $L$, so the
filter can be applied with $K$ sparse matrix-vector products and no eigendecomposition.

In [32]:
Af = np.array([[0.0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])
Lf = np.diag(Af.sum(1)) - Af
wf, Uf = np.linalg.eigh(Lf)
xf = rng.standard_normal(4)
gfun = lambda lam: np.exp(-0.5 * lam)
spectral = Uf @ np.diag(gfun(wf)) @ Uf.T @ xf
direct = sla.expm(-0.5 * Lf) @ xf
print("filtered via GFT :", spectral)
print("expm(-0.5 L) x   :", direct)
poly = np.eye(4) - 0.5 * Lf + 0.125 * Lf @ Lf
print("degree-2 Taylor filter:", poly @ xf)
assert np.allclose(spectral, direct)

filtered via GFT : [ 0.2488 -0.402   0.0369  0.5766]
expm(-0.5 L) x   : [ 0.2488 -0.402   0.0369  0.5766]
degree-2 Taylor filter: [ 0.4642 -0.755   0.1763  0.575 ]


### Problem L2.7 — Chebyshev graph convolution and its cost

**Statement.** For $g_\theta(L)x = \sum_{k=0}^{K}\theta_k T_k(\tilde{L})x$ with
$\tilde{L} = \tfrac{2}{\lambda_n}L - I$, compute $\partial (g_\theta(L)x)/\partial\theta_k$ and
give the cost of evaluating the filter.

**Intuition.** The filter is linear in the coefficients, and the Chebyshev recurrence turns the
whole sum into repeated sparse multiplications.

**Solution.**

*Step 1.* Linearity in $\theta$ gives
$\partial (g_\theta(L)x)/\partial\theta_k = T_k(\tilde{L})x$.

*Step 2.* The recurrence $T_0 = I$, $T_1(\tilde{L}) = \tilde{L}$,
$T_{k}(\tilde{L}) = 2\tilde{L}T_{k-1}(\tilde{L}) - T_{k-2}(\tilde{L})$ acts on the vector
$x_k = T_k(\tilde{L})x$ as $x_k = 2\tilde{L}x_{k-1} - x_{k-2}$.

*Step 3.* $\tilde{L}$ has $O(\lvert E \rvert)$ non-zeros, so each recurrence step costs
$O(\lvert E \rvert)$ and the whole filter costs $O(K\lvert E \rvert)$, against $O(n^3)$ for an
eigendecomposition.

$$
\boxed{\frac{\partial (g_\theta(L)x)}{\partial\theta_k} = T_k(\tilde{L})x, \qquad \text{cost } O(K\lvert E \rvert)}
$$

**Key takeaway.** $T_k(\tilde{L})$ has zero entries beyond graph distance $k$, so a degree-$K$
Chebyshev filter is exactly $K$-hop local — depth and receptive field are the same parameter.

In [33]:
nch = 12
Ach = np.zeros((nch, nch))
for i in range(nch - 1):
    Ach[i, i + 1] = Ach[i + 1, i] = 1.0
Lch = np.diag(Ach.sum(1)) - Ach
lam_max = np.linalg.eigvalsh(Lch)[-1]
Lt = 2.0 / lam_max * Lch - np.eye(nch)
xch = rng.standard_normal(nch)
K = 3
theta = rng.standard_normal(K + 1)
xs_ = [xch, Lt @ xch]
for k in range(2, K + 1):
    xs_.append(2 * Lt @ xs_[k - 1] - xs_[k - 2])
y_rec = sum(theta[k] * xs_[k] for k in range(K + 1))
Tk = [np.eye(nch), Lt]
for k in range(2, K + 1):
    Tk.append(2 * Lt @ Tk[k - 1] - Tk[k - 2])
y_mat = sum(theta[k] * Tk[k] @ xch for k in range(K + 1))
print("recurrence vs explicit polynomial:", np.abs(y_rec - y_mat).max())
print("gradient wrt theta_2 equals T_2(Lt) x:", np.abs(xs_[2] - Tk[2] @ xch).max())
band = np.abs(Tk[K]) > 1e-12
print(f"T_{K}(Lt) is zero beyond graph distance {K}:",
      all(not band[i, j] for i in range(nch) for j in range(nch) if abs(i - j) > K))
assert np.abs(y_rec - y_mat).max() < 1e-10
assert all(not band[i, j] for i in range(nch) for j in range(nch) if abs(i - j) > K)

recurrence vs explicit polynomial: 8.881784197001252e-16
gradient wrt theta_2 equals T_2(Lt) x: 2.220446049250313e-16
T_3(Lt) is zero beyond graph distance 3: True


### Problem L2.8 — Gradient of the attention logits

**Statement.** In scaled dot-product attention, $S = QK^{\top}/\sqrt{d_k}$ and $A = \operatorname{softmax}(S)$
row by row. Given $G = \nabla_A J$, derive $\nabla_Q J$ and $\nabla_K J$.

**Intuition.** Each row of $A$ is an independent softmax, so the row-wise Jacobian of Problem L2.1
applies once per row and then the linear map $Q \mapsto QK^{\top}$ is transposed.

**Solution.**

*Step 1.* Applying Problem L2.1 to row $i$ of $S$,

$$
(\nabla_S J)_{ij} = A_{ij}\Bigl( G_{ij} - \sum_{l} G_{il}A_{il} \Bigr),
$$

which in matrix form is $\Gamma := A \odot \bigl( G - (G \odot A)\mathbf{1}\mathbf{1}^{\top} \bigr)$.

*Step 2.* From $S = QK^{\top}/\sqrt{d_k}$ with $K$ fixed,
$dS = (dQ)K^{\top}/\sqrt{d_k}$, so

$$
dJ = \operatorname{tr}\bigl( \Gamma^{\top}(dQ)K^{\top} \bigr)/\sqrt{d_k}
= \operatorname{tr}\Bigl( \bigl( \Gamma K/\sqrt{d_k} \bigr)^{\top}dQ \Bigr) .
$$

*Step 3.* The same computation with $Q$ fixed gives the gradient in $K$.

$$
\boxed{\nabla_Q J = \frac{\Gamma K}{\sqrt{d_k}}, \qquad \nabla_K J = \frac{\Gamma^{\top}Q}{\sqrt{d_k}}, \qquad \Gamma = A \odot \bigl( G - (G\odot A)\mathbf{1}\mathbf{1}^{\top} \bigr)}
$$

**Key takeaway.** The bracket in $\Gamma$ subtracts a row-wise weighted mean: softmax gradients
are always centred, which is the discrete trace of $J_s\mathbf{1} = 0$.

In [34]:
N_, dk = 5, 3
Q = rng.standard_normal((N_, dk))
Kk = rng.standard_normal((N_, dk))
Tatt = rng.standard_normal((N_, N_))


def attn_loss(Q_, K_):
    S = Q_ @ K_.T / np.sqrt(dk)
    S = S - S.max(axis=1, keepdims=True)
    Aa = np.exp(S)
    Aa = Aa / Aa.sum(axis=1, keepdims=True)
    return 0.5 * np.sum((Aa - Tatt) ** 2), Aa


J0, Aa = attn_loss(Q, Kk)
G = Aa - Tatt
Gam = Aa * (G - (G * Aa).sum(axis=1, keepdims=True))
gQ = Gam @ Kk / np.sqrt(dk)
gK = Gam.T @ Q / np.sqrt(dk)
h = 1e-6
fdQ = np.zeros_like(Q)
for idx in np.ndindex(Q.shape):
    E = np.zeros_like(Q)
    E[idx] = h
    fdQ[idx] = (attn_loss(Q + E, Kk)[0] - attn_loss(Q - E, Kk)[0]) / (2 * h)
fdK = np.zeros_like(Kk)
for idx in np.ndindex(Kk.shape):
    E = np.zeros_like(Kk)
    E[idx] = h
    fdK[idx] = (attn_loss(Q, Kk + E)[0] - attn_loss(Q, Kk - E)[0]) / (2 * h)
print("max |grad_Q - finite difference| =", np.abs(fdQ - gQ).max())
print("max |grad_K - finite difference| =", np.abs(fdK - gK).max())
print("row sums of Gamma (should be 0):", Gam.sum(axis=1))
assert np.abs(fdQ - gQ).max() < 1e-6 and np.abs(fdK - gK).max() < 1e-6
assert np.abs(Gam.sum(axis=1)).max() < 1e-12

max |grad_Q - finite difference| = 1.2010775429782683e-09
max |grad_K - finite difference| = 2.5255553204317494e-09
row sums of Gamma (should be 0): [ 0.  0.  0. -0.  0.]


### Problem L2.9 — The PageRank Google matrix

**Statement.** For column-stochastic $P$ and $\alpha \in (0,1)$, show that
$M = (1-\alpha)P + \tfrac{\alpha}{n}\mathbf{1}\mathbf{1}^{\top}$ is column-stochastic and primitive,
and bound the modulus of every eigenvalue other than $1$.

**Intuition.** Teleportation mixes a little of the uniform distribution into every column, which
makes the matrix strictly positive and hence primitive.

**Solution.**

*Step 1.* $\mathbf{1}^{\top}M = (1-\alpha)\mathbf{1}^{\top}P + \tfrac{\alpha}{n}(\mathbf{1}^{\top}\mathbf{1})\mathbf{1}^{\top}
= (1-\alpha)\mathbf{1}^{\top} + \alpha\mathbf{1}^{\top} = \mathbf{1}^{\top}$.

*Step 2.* Every entry satisfies $M_{ij} \ge \alpha/n \gt 0$, so $M \gt 0$ and $M$ is primitive
with $k = 1$.

*Step 3.* Let $Mv = \lambda v$ with $\lambda \neq 1$. Then $\mathbf{1}^{\top}v = 0$, because
$\lambda \mathbf{1}^{\top}v = \mathbf{1}^{\top}Mv = \mathbf{1}^{\top}v$. On that subspace the
rank-one term vanishes, $\mathbf{1}\mathbf{1}^{\top}v = 0$, so $Mv = (1-\alpha)Pv$ and
$\lvert \lambda \rvert \le (1-\alpha)\rho(P) = 1-\alpha$ by Problem L1.14.

$$
\boxed{M \text{ is column-stochastic and primitive}, \qquad \lvert \lambda \rvert \le 1-\alpha \text{ for } \lambda \neq 1}
$$

**Key takeaway.** The damping factor buys convergence: Theorem 4.8 gives a per-step contraction of
at most $1-\alpha = 0.85$ at the standard $\alpha = 0.15$, whatever the link graph looks like.

In [35]:
nP = 6
links = {0: [1, 2], 1: [2], 2: [0, 3], 3: [4], 4: [2, 5], 5: [2]}
Pp = np.zeros((nP, nP))
for j, outs in links.items():
    for i in outs:
        Pp[i, j] = 1.0 / len(outs)
alpha = 0.15
M = (1 - alpha) * Pp + alpha / nP * np.ones((nP, nP))
ev = np.linalg.eigvals(M)
lam2 = np.sort(np.abs(ev))[-2]
print("column sums:", M.sum(axis=0), "   min entry:", M.min())
print("eigenvalue moduli:", np.sort(np.abs(ev))[::-1])
print(f"|lambda_2| = {lam2:.6f}   bound 1 - alpha = {1 - alpha:.2f}")
r = np.ones(nP) / nP
for _ in range(200):
    r = M @ r
print("PageRank vector:", r, "  sum =", r.sum())
assert np.allclose(M.sum(axis=0), 1.0) and M.min() > 0
assert lam2 <= 1 - alpha + 1e-12
assert np.linalg.norm(M @ r - r) < 1e-12

column sums: [1. 1. 1. 1. 1. 1.]    min entry: 0.024999999999999998
eigenvalue moduli: [1.    0.601 0.601 0.425 0.    0.   ]
|lambda_2| = 0.601041   bound 1 - alpha = 0.85
PageRank vector: [0.1625 0.0941 0.3235 0.1625 0.1631 0.0943]   sum = 1.0000000000000007


### Problem L2.10 — The Fiedler vector as a relaxed minimum cut

**Statement.** Show that the Fiedler vector solves

$$
\min_{x \neq 0, \ x \perp \mathbf{1}} \ \frac{\sum_{\lbrace i,j \rbrace \in E}A_{ij}(x_i - x_j)^2}{\lVert x \rVert_2^2},
$$

with optimal value $\lambda_2(L)$.

**Intuition.** The numerator is the Dirichlet energy of Problem L1.9, so the ratio is a Rayleigh
quotient and the constraint removes the trivial constant minimizer.

**Solution.**

*Step 1.* By Problem L1.9 the objective is $R(x) = x^{\top}Lx / x^{\top}x$.

*Step 2.* $L$ is symmetric, so it has an orthonormal eigenbasis $\phi_1 = \mathbf{1}/\sqrt{n}, \phi_2, \dots$
with $0 = \lambda_1 \le \lambda_2 \le \cdots$.

*Step 3.* For $x = \sum_{k \ge 2}c_k\phi_k$ orthogonal to $\mathbf{1}$,

$$
R(x) = \frac{\sum_{k \ge 2}\lambda_kc_k^2}{\sum_{k \ge 2}c_k^2} \ \ge \ \lambda_2 ,
$$

with equality at $x = \phi_2$.

$$
\boxed{\min_{x \perp \mathbf{1}, \ x \neq 0} R(x) = \lambda_2(L), \text{ attained at the Fiedler vector } \phi_2}
$$

**Key takeaway.** Thresholding $\phi_2$ at zero is the standard rounding of this relaxation, and
Section 7.5 of the theory notebook shows it recovering the exact optimal cut.

In [36]:
nb_ = 8
Ab = np.zeros((nb_, nb_))
for blk in (range(0, 4), range(4, 8)):
    for i in blk:
        for j in blk:
            if i != j:
                Ab[i, j] = 1.0
Ab[3, 4] = Ab[4, 3] = 1.0
Lb = np.diag(Ab.sum(1)) - Ab
wb, Vb = np.linalg.eigh(Lb)
phi2 = Vb[:, 1]
print("lambda_2 =", wb[1], "   R(phi2) =", phi2 @ Lb @ phi2 / (phi2 @ phi2))
best = np.inf
for _ in range(2000):
    xr = rng.standard_normal(nb_)
    xr = xr - xr.mean()
    best = min(best, xr @ Lb @ xr / (xr @ xr))
print("best Rayleigh quotient from 2000 random x perpendicular to 1:", best)
print("Fiedler signs:", np.sign(phi2).astype(int))
assert abs(phi2 @ Lb @ phi2 / (phi2 @ phi2) - wb[1]) < 1e-12
assert best >= wb[1] - 1e-12

lambda_2 = 0.35424868893540884    R(phi2) = 0.3542486889354094
best Rayleigh quotient from 2000 random x perpendicular to 1: 0.540264201020982
Fiedler signs: [ 1  1  1  1 -1 -1 -1 -1]


### Problem L2.11 — A neural graph ODE through the Kronecker product

**Statement.** Solve $\dot{H}(t) = -L_{\mathrm{sym}}H(t)W$ with $H(t) \in \mathbb{R}^{N \times d}$
and constant $W \in \mathbb{R}^{d \times d}$.

**Intuition.** The equation is linear but acts on both sides of $H$; vectorizing turns it into an
ordinary linear system.

**Solution.**

*Step 1.* Apply $\operatorname{vec}$ and Theorem 4.3 to the right-hand side:

$$
\operatorname{vec}\bigl( L_{\mathrm{sym}}HW \bigr) = \bigl( W^{\top}\otimes L_{\mathrm{sym}} \bigr)\operatorname{vec}(H) .
$$

*Step 2.* With $h(t) = \operatorname{vec}(H(t))$ the system is
$\dot{h} = -\bigl( W^{\top}\otimes L_{\mathrm{sym}} \bigr)h$.

*Step 3.* Theorem 4.7 solves it.

$$
\boxed{\operatorname{vec}(H(t)) = e^{-t(W^{\top}\otimes L_{\mathrm{sym}})}\operatorname{vec}(H(0))}
$$

**Key takeaway.** By Proof 5.3 Step 3 the spectrum of $W^{\top}\otimes L_{\mathrm{sym}}$ is the set
of products $\mu_j\lambda_i$, so the layer is stable exactly when every such product has
non-negative real part.

In [37]:
Nn, dd = 5, 2
Aq = np.array([[0.0, 1, 0, 0, 1], [1, 0, 1, 0, 0], [0, 1, 0, 1, 0],
               [0, 0, 1, 0, 1], [1, 0, 0, 1, 0]])
dq = Aq.sum(1)
Lq = np.diag(dq ** -0.5) @ (np.diag(dq) - Aq) @ np.diag(dq ** -0.5)
Wq = np.array([[1.0, 0.3], [0.0, 0.5]])
H0 = rng.standard_normal((Nn, dd))
tq = 0.6
h_sol = sla.expm(-tq * np.kron(Wq.T, Lq)) @ vec(H0)
H_sol = h_sol.reshape(Nn, dd, order="F")


from scipy.integrate import solve_ivp

sol = solve_ivp(lambda t, h: -np.kron(Wq.T, Lq) @ h, (0.0, tq), vec(H0),
                rtol=1e-11, atol=1e-13)
H_num = sol.y[:, -1].reshape(Nn, dd, order="F")
print("Kronecker solution:\n", H_sol)
print("numerical integration of the ODE:\n", H_num)
print("max difference:", np.abs(H_sol - H_num).max())
print("spectrum of W^T kron L (first six):",
      np.sort(np.linalg.eigvals(np.kron(Wq.T, Lq)).real)[:6])
assert np.abs(H_sol - H_num).max() < 1e-8

Kronecker solution:
 [[ 0.3853 -0.2733]
 [ 0.0484 -0.7618]
 [ 0.2923 -0.6476]
 [ 0.1549  0.6727]
 [ 0.164   0.5164]]
numerical integration of the ODE:
 [[ 0.3853 -0.2733]
 [ 0.0484 -0.7618]
 [ 0.2923 -0.6476]
 [ 0.1549  0.6727]
 [ 0.164   0.5164]]
max difference: 2.772365670367094e-13
spectrum of W^T kron L (first six): [-0.      0.      0.3455  0.3455  0.691   0.691 ]


### Problem L2.12 — Physics: a driven linear system

**Statement.** Solve $\dot{x}(t) = Ax(t) + f(t)$ with $x(0) = x_0$, and evaluate it for the driven
undamped oscillator $A = \left[\begin{smallmatrix}0&1\\-\omega^2&0\end{smallmatrix}\right]$,
$f(t) = (0, F)^{\top}$ constant.

**Intuition.** Multiply by the integrating factor $e^{-At}$ so the left side becomes an exact
derivative, exactly as in the scalar case.

**Solution.**

*Step 1.* Multiply $\dot{x} - Ax = f$ on the left by $e^{-At}$ and use Theorem 4.7:

$$
\frac{d}{dt}\bigl( e^{-At}x(t) \bigr) = e^{-At}\dot{x} - Ae^{-At}x = e^{-At}f(t) .
$$

*Step 2.* Integrate from $0$ to $t$ and multiply by $e^{At}$:

$$
x(t) = e^{At}x_0 + \int_0^t e^{A(t-\tau)}f(\tau)\,d\tau .
$$

*Step 3.* For the oscillator with constant force, $A$ is invertible with
$A^{-1} = \left[\begin{smallmatrix}0 & -\omega^{-2} \\ 1 & 0\end{smallmatrix}\right]$, so
$\int_0^t e^{A(t-\tau)}f\,d\tau = A^{-1}\bigl( e^{At} - I \bigr)f$. Writing
$f = (0, F)^{\top}$ and $e^{At} = \left[\begin{smallmatrix}\cos\omega t & \omega^{-1}\sin\omega t \\ -\omega\sin\omega t & \cos\omega t\end{smallmatrix}\right]$
gives the displacement $q(t) = q_0\cos\omega t + \tfrac{\dot{q}_0}{\omega}\sin\omega t + \tfrac{F}{\omega^2}\bigl( 1 - \cos\omega t \bigr)$.

$$
\boxed{x(t) = e^{At}x_0 + \int_0^t e^{A(t-\tau)}f(\tau)\,d\tau, \qquad q(t) = q_0\cos\omega t + \frac{\dot q_0}{\omega}\sin\omega t + \frac{F}{\omega^2}(1 - \cos\omega t)}
$$

**Key takeaway.** A constant force shifts the equilibrium to $F/\omega^2$ and the mass oscillates
about the new equilibrium with the same frequency — the physical content of the particular
solution.

In [38]:
omega, F = 2.0, 3.0
Aosc = np.array([[0.0, 1.0], [-omega ** 2, 0.0]])
fvec = np.array([0.0, F])
x0 = np.array([0.5, -1.0])
tq = 1.7
closed = sla.expm(Aosc * tq) @ x0 + np.linalg.solve(Aosc, (sla.expm(Aosc * tq) - np.eye(2)) @ fvec)
q_formula = (x0[0] * np.cos(omega * tq) + x0[1] / omega * np.sin(omega * tq)
             + F / omega ** 2 * (1 - np.cos(omega * tq)))
from scipy.integrate import solve_ivp

sol = solve_ivp(lambda t, x: Aosc @ x + fvec, (0.0, tq), x0, rtol=1e-11, atol=1e-13)
xn = sol.y[:, -1]
print("variation of parameters :", closed)
print("scalar formula for q(t) :", q_formula)
print("numerical integration   :", xn)
assert abs(closed[0] - q_formula) < 1e-10
assert np.abs(closed - xn).max() < 1e-7

variation of parameters : [1.1195 0.839 ]
scalar formula for q(t) : 1.1194700991582809
numerical integration   : [1.1195 0.839 ]


### Problem L2.13 — Physics: effective resistance in a resistor network

**Statement.** Model a connected graph as a network of unit conductances. Show that the effective
resistance between $i$ and $j$ is
$R_{ij} = (e_i - e_j)^{\top}L^{+}(e_i - e_j) = L^{+}_{ii} + L^{+}_{jj} - 2L^{+}_{ij}$.

**Intuition.** Kirchhoff's current law at every node is exactly $Lv = i_{\mathrm{ext}}$, so
inverting $L$ on the space of balanced currents gives potentials.

**Solution.**

*Step 1.* Inject one ampere at $i$ and extract it at $j$: $i_{\mathrm{ext}} = e_i - e_j$, and node
potentials satisfy $Lv = i_{\mathrm{ext}}$.

*Step 2.* $\mathbf{1}^{\top}i_{\mathrm{ext}} = 0$, so $i_{\mathrm{ext}} \perp \operatorname{Null}(L)$
and the general solution is $v = L^{+}i_{\mathrm{ext}} + c\mathbf{1}$.

*Step 3.* The measured voltage is $v_i - v_j = (e_i - e_j)^{\top}v$, and
$(e_i - e_j)^{\top}\mathbf{1} = 0$ kills the constant, so the answer does not depend on the choice
of ground.

*Step 4.* Ohm's law with unit current makes the resistance equal the voltage, and expanding the
quadratic form gives the entrywise expression.

$$
\boxed{R_{ij} = (e_i - e_j)^{\top}L^{+}(e_i - e_j) = L^{+}_{ii} + L^{+}_{jj} - 2L^{+}_{ij}}
$$

**Key takeaway.** Effective resistance is a metric on the vertices, and for a path of $k$ unit
resistors in series it returns exactly $k$ — the code cell checks that.

In [39]:
npath = 5
Apath = np.zeros((npath, npath))
for i in range(npath - 1):
    Apath[i, i + 1] = Apath[i + 1, i] = 1.0
Lpath = np.diag(Apath.sum(1)) - Apath
Lp = np.linalg.pinv(Lpath)
for (i, j) in [(0, 1), (0, 2), (0, 4)]:
    e = np.zeros(npath)
    e[i], e[j] = 1.0, -1.0
    R = e @ Lp @ e
    print(f"R({i},{j}) = {R:.6f}   entrywise form {Lp[i,i] + Lp[j,j] - 2*Lp[i,j]:.6f}"
          f"   series prediction {abs(i - j)}")
    assert abs(R - abs(i - j)) < 1e-10
Asq = np.array([[0.0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])
Lsq = np.diag(Asq.sum(1)) - Asq
Lsqp = np.linalg.pinv(Lsq)
e = np.zeros(4)
e[0], e[2] = 1.0, -1.0
print(f"4-cycle, opposite corners: R = {e @ Lsqp @ e:.6f}   (two 2-ohm paths in parallel = 1)")
assert abs(e @ Lsqp @ e - 1.0) < 1e-10

R(0,1) = 1.000000   entrywise form 1.000000   series prediction 1
R(0,2) = 2.000000   entrywise form 2.000000   series prediction 2
R(0,4) = 4.000000   entrywise form 4.000000   series prediction 4
4-cycle, opposite corners: R = 1.000000   (two 2-ohm paths in parallel = 1)


### Problem L2.14 — Physics: the heat kernel trace counts edges

**Statement.** For an unweighted graph with Laplacian spectrum
$0 = \lambda_1 \le \cdots \le \lambda_n$, show
$\operatorname{tr}(e^{-tL}) = \sum_i e^{-t\lambda_i}$ and expand it as $t \to 0^{+}$.

**Intuition.** The heat trace measures how much heat has stayed where it started; at short times
that is controlled by how many neighbours each vertex has.

**Solution.**

*Step 1.* $L = U\Lambda U^{\top}$ gives $e^{-tL} = Ue^{-t\Lambda}U^{\top}$, and cyclicity of the
trace removes $U$: $\operatorname{tr}(e^{-tL}) = \operatorname{tr}(e^{-t\Lambda})$.

*Step 2.* Expand each scalar exponential: $e^{-t\lambda_i} = 1 - t\lambda_i + O(t^2)$.

*Step 3.* Sum and use $\sum_i\lambda_i = \operatorname{tr}(L) = 2\lvert E \rvert$ from Problem L0.8.

$$
\boxed{\operatorname{tr}(e^{-tL}) = \sum_{i=1}^{n}e^{-t\lambda_i} = n - 2\lvert E \rvert t + O(t^2)}
$$

**Key takeaway.** The short-time heat trace recovers the vertex count and then the edge count —
the graph analogue of Weyl's asymptotics for the Laplacian on a manifold.

In [40]:
Ahk = np.array([[0.0, 1, 1, 0, 0], [1, 0, 1, 0, 0], [1, 1, 0, 1, 0],
                [0, 0, 1, 0, 1], [0, 0, 0, 1, 0]])
Lhk = np.diag(Ahk.sum(1)) - Ahk
nhk = 5
Ehk = int(Ahk.sum() / 2)
lam = np.linalg.eigvalsh(Lhk)
for t in (0.2, 0.05, 0.01, 0.002):
    tr = np.trace(sla.expm(-t * Lhk))
    lin = nhk - 2 * Ehk * t
    print(f"t={t:6.3f}   tr(e^-tL) = {tr:.8f}   n - 2|E| t = {lin:.8f}"
          f"   residual/t^2 = {(tr - lin)/t**2:.4f}")
    assert abs(tr - sum(np.exp(-t * lam))) < 1e-10
print(f"n = {nhk}, |E| = {Ehk}, trace(L) = {np.trace(Lhk):.0f} = 2|E|")
assert abs(np.trace(Lhk) - 2 * Ehk) < 1e-12

t= 0.200   tr(e^-tL) = 3.51443605   n - 2|E| t = 3.00000000   residual/t^2 = 12.8609
t= 0.050   tr(e^-tL) = 4.53777000   n - 2|E| t = 4.50000000   residual/t^2 = 15.1080
t= 0.010   tr(e^-tL) = 4.90158150   n - 2|E| t = 4.90000000   residual/t^2 = 15.8150
t= 0.002   tr(e^-tL) = 4.98006385   n - 2|E| t = 4.98000000   residual/t^2 = 15.9627
n = 5, |E| = 5, trace(L) = 10 = 2|E|


## L3 — Challenge Proofs

### Problem L3.1 — Uniqueness of the stationary distribution

**Statement.** Let $P$ be column-stochastic and primitive. Prove that there is exactly one $\pi$
with $P\pi = \pi$, $\pi \gt 0$ entrywise, and $\mathbf{1}^{\top}\pi = 1$.

**Intuition.** Primitivity means every state reaches every other in the same number of steps, so
no probability can pool anywhere or cycle forever.

**Solution.**

*Step 1.* $\rho(P) = 1$. Column-stochasticity gives $\mathbf{1}^{\top}P = \mathbf{1}^{\top}$, so
$1 \in \operatorname{spec}(P^{\top}) = \operatorname{spec}(P)$ and $\rho(P) \ge 1$. Conversely
$\lVert P \rVert_1 = \max_j \sum_i \lvert P_{ij} \rvert = 1$ and $\rho(P) \le \lVert P \rVert_1$,
so $\rho(P) = 1$ exactly. This second half is the one usually skipped.

*Step 2.* Theorem 4.8b (Perron-Frobenius for primitive matrices) applies with $\rho(P) = 1$: the
eigenvalue $1$ is algebraically simple, its eigenvector may be taken strictly positive, and every
other eigenvalue has modulus strictly below $1$.

*Step 3.* Simplicity makes the eigenspace one-dimensional, so any solution of $Pv = v$ is a scalar
multiple of the Perron vector.

*Step 4.* Exactly one multiple satisfies $\mathbf{1}^{\top}\pi = 1$, and since the Perron vector is
strictly positive that multiple is positive, so $\pi \gt 0$.

$$
\boxed{P\pi = \pi, \quad \pi \gt 0, \quad \mathbf{1}^{\top}\pi = 1 \ \text{ has exactly one solution}}
$$

**Key takeaway.** Existence needs only stochasticity; uniqueness needs irreducibility; convergence
to $\pi$ needs primitivity, as the periodic counterexample in Section 7.3 of the theory notebook
shows.

In [41]:
Pr = np.array([[0.5, 0.2, 0.1], [0.3, 0.6, 0.3], [0.2, 0.2, 0.6]])
print("column sums:", Pr.sum(axis=0), "   min entry:", Pr.min())
print("||P||_1 =", np.linalg.norm(Pr, 1), "   spectral radius =", np.abs(np.linalg.eigvals(Pr)).max())
w, V = np.linalg.eig(Pr)
i1 = int(np.argmin(np.abs(w - 1.0)))
pi = np.real(V[:, i1])
pi = pi / pi.sum()
print("stationary pi :", pi, "  all positive:", (pi > 0).all())
print("eigenvalue moduli:", np.sort(np.abs(w))[::-1])
print("dimension of the eigenspace for 1 :", 3 - np.linalg.matrix_rank(Pr - np.eye(3)))
assert np.linalg.norm(Pr @ pi - pi) < 1e-12 and (pi > 0).all()
assert 3 - np.linalg.matrix_rank(Pr - np.eye(3)) == 1

column sums: [1. 1. 1.]    min entry: 0.1
||P||_1 = 1.0    spectral radius = 1.0
stationary pi : [0.2381 0.4286 0.3333]   all positive: True
eigenvalue moduli: [1.  0.4 0.3]
dimension of the eigenspace for 1 : 1


### Problem L3.2 — Frechet derivative of the exponential, commuting case

**Statement.** For $X, V \in \mathbb{R}^{n \times n}$ with $XV = VX$, compute
$De^{X}(V) = \left.\dfrac{d}{dt}e^{X+tV}\right|_{t=0}$.

**Intuition.** When the two matrices commute, everything behaves as it does for scalars.

**Solution.**

*Step 1.* Expand a power to first order in $t$:

$$
(X + tV)^k = X^k + t\sum_{j=0}^{k-1}X^{j}VX^{k-1-j} + O(t^2) .
$$

*Step 2.* Commutation collapses every summand: $X^{j}VX^{k-1-j} = VX^{k-1}$, so the sum is
$kVX^{k-1}$.

*Step 3.* Differentiate the series term by term, which is legitimate by the uniform convergence of
Proof 5.7 Step 1:

$$
De^{X}(V) = \sum_{k=1}^{\infty}\frac{kVX^{k-1}}{k!} = V\sum_{m=0}^{\infty}\frac{X^{m}}{m!} = Ve^{X} .
$$

$$
\boxed{De^{X}(V) = Ve^{X} = e^{X}V \quad \text{when } XV = VX}
$$

**Key takeaway.** Without commutation the answer is the integral
$\int_0^1 e^{(1-s)X}Ve^{sX}\,ds$, which reduces to $Ve^{X}$ exactly when the integrand is constant
in $s$ — that is, exactly when $X$ and $V$ commute.

In [42]:
Xc = np.array([[1.0, 2.0], [0.0, 1.0]])
Vc = np.array([[3.0, 1.0], [0.0, 3.0]])          # commutes with Xc
print("commutator XV - VX:\n", Xc @ Vc - Vc @ Xc)
h = 1e-6
fd = (sla.expm(Xc + h * Vc) - sla.expm(Xc - h * Vc)) / (2 * h)
print("finite difference :\n", fd)
print("V e^X             :\n", Vc @ sla.expm(Xc))
ss = np.linspace(0, 1, 4001)
integral = np.trapezoid(np.array([sla.expm((1 - s) * Xc) @ Vc @ sla.expm(s * Xc) for s in ss]),
                        ss, axis=0)
print("integral formula  :\n", integral)
Vnc = np.array([[0.0, 0.0], [1.0, 0.0]])         # does not commute
fd_nc = (sla.expm(Xc + h * Vnc) - sla.expm(Xc - h * Vnc)) / (2 * h)
print("non-commuting case: ||fd - V e^X||_F =", np.linalg.norm(fd_nc - Vnc @ sla.expm(Xc)))
assert np.abs(fd - Vc @ sla.expm(Xc)).max() < 1e-6
assert np.abs(integral - Vc @ sla.expm(Xc)).max() < 1e-6
assert np.linalg.norm(fd_nc - Vnc @ sla.expm(Xc)) > 0.5

commutator XV - VX:
 [[0. 0.]
 [0. 0.]]
finite difference :
 [[ 8.1548 19.028 ]
 [ 0.      8.1548]]
V e^X             :
 [[ 8.1548 19.028 ]
 [ 0.      8.1548]]
integral formula  :
 [[ 8.1548 19.028 ]
 [ 0.      8.1548]]
non-commuting case: ||fd - V e^X||_F = 4.24995730985094


### Problem L3.3 — The Lie-Trotter product formula

**Statement.** For any $A, B \in \mathbb{R}^{n \times n}$, prove
$e^{A+B} = \lim_{k \to \infty}\bigl( e^{A/k}e^{B/k} \bigr)^{k}$, with error $O(1/k)$.

**Intuition.** Over one short step the commutator defect is $O(1/k^2)$, and there are only $k$
steps, so the total defect is $O(1/k)$.

**Solution.**

*Step 1 — one-step defect.* Expanding both sides to second order,

$$
e^{A/k}e^{B/k} = I + \frac{A+B}{k} + \frac{A^2 + 2AB + B^2}{2k^2} + O(k^{-3}),
\qquad
e^{(A+B)/k} = I + \frac{A+B}{k} + \frac{(A+B)^2}{2k^2} + O(k^{-3}),
$$

so with $E_k = e^{A/k}e^{B/k}$ and $F_k = e^{(A+B)/k}$,

$$
E_k - F_k = \frac{[A,B]}{2k^2} + O(k^{-3}), \qquad [A,B] = AB - BA .
$$

*Step 2 — a uniform norm bound.* With $c = \lVert A \rVert_{\mathrm{op}} + \lVert B \rVert_{\mathrm{op}}$,
both $\lVert E_k \rVert_{\mathrm{op}}$ and $\lVert F_k \rVert_{\mathrm{op}}$ are at most $e^{c/k}$,
by the series bound of Proof 5.7 Step 1.

*Step 3 — telescoping.* For any two matrices,

$$
E_k^{\,k} - F_k^{\,k} = \sum_{j=0}^{k-1} E_k^{\,j}\bigl( E_k - F_k \bigr)F_k^{\,k-1-j} .
$$

Taking norms and using Step 2,

$$
\bigl\lVert E_k^{\,k} - F_k^{\,k} \bigr\rVert_{\mathrm{op}} \ \le \ k \cdot e^{c(k-1)/k} \cdot \lVert E_k - F_k \rVert_{\mathrm{op}}
\ \le \ \frac{C}{k} .
$$

*Step 4 — conclude.* $F_k^{\,k} = e^{A+B}$ exactly, because $(A+B)/k$ commutes with itself.
Letting $k \to \infty$ gives the limit, with the stated first-order rate.

$$
\boxed{e^{A+B} = \lim_{k \to \infty}\bigl( e^{A/k}e^{B/k} \bigr)^{k}, \qquad \text{error } = O(1/k)}
$$

**Key takeaway.** Section 7.2 of the theory notebook measures the order and gets $0.9880$ against
the predicted $1$. The symmetric variant $\bigl( e^{A/2k}e^{B/k}e^{A/2k} \bigr)^{k}$ is second
order, which is why every practical splitting integrator uses it.

In [43]:
A = np.array([[0.0, 1.0], [0.0, 0.0]])
B = np.array([[0.0, 0.0], [1.0, 0.0]])
target = sla.expm(A + B)
ks = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512])
err_lt = np.array([np.linalg.norm(np.linalg.matrix_power(sla.expm(A / k) @ sla.expm(B / k), k) - target)
                   for k in ks])
err_st = np.array([np.linalg.norm(np.linalg.matrix_power(
    sla.expm(A / (2 * k)) @ sla.expm(B / k) @ sla.expm(A / (2 * k)), k) - target) for k in ks])
o_lt = -np.polyfit(np.log(ks), np.log(err_lt), 1)[0]
o_st = -np.polyfit(np.log(ks), np.log(err_st), 1)[0]
for k, a_, b_ in zip(ks, err_lt, err_st):
    print(f"  k={k:4d}   Lie-Trotter {a_:.3e}   Strang {b_:.3e}")
print(f"observed orders: Lie-Trotter {o_lt:.4f} (predicted 1), Strang {o_st:.4f} (predicted 2)")
assert 0.9 < o_lt < 1.1 and 1.9 < o_st < 2.1

  k=   1   Lie-Trotter 7.517e-01   Strang 2.000e-01
  k=   2   Lie-Trotter 4.044e-01   Strang 5.661e-02
  k=   4   Lie-Trotter 2.063e-01   Strang 1.465e-02
  k=   8   Lie-Trotter 1.037e-01   Strang 3.694e-03
  k=  16   Lie-Trotter 5.191e-02   Strang 9.255e-04
  k=  32   Lie-Trotter 2.597e-02   Strang 2.315e-04
  k=  64   Lie-Trotter 1.298e-02   Strang 5.789e-05
  k= 128   Lie-Trotter 6.492e-03   Strang 1.447e-05
  k= 256   Lie-Trotter 3.246e-03   Strang 3.618e-06
  k= 512   Lie-Trotter 1.623e-03   Strang 9.045e-07
observed orders: Lie-Trotter 0.9901 (predicted 1), Strang 1.9832 (predicted 2)


### Problem L3.4 — Gradient of the nuclear norm

**Statement.** Let $X \in \mathbb{R}^{m \times n}$, $m \ge n$, have $n$ distinct strictly positive
singular values, with thin SVD $X = U\Sigma V^{\top}$. Prove
$\nabla_X \lVert X \rVert_{*} = UV^{\top}$, where $\lVert X \rVert_{*} = \sum_i\sigma_i$.

**Intuition.** The nuclear norm is the $\ell_1$ norm of the singular values, so its gradient
should be the matrix analogue of the sign function — and $UV^{\top}$ is exactly the orthogonal
polar factor of $X$.

**Solution.**

*Step 1 — differentiate one singular value.* From $\sigma_i = u_i^{\top}Xv_i$ with
$\lVert u_i \rVert = \lVert v_i \rVert = 1$,

$$
d\sigma_i = (du_i)^{\top}Xv_i + u_i^{\top}(dX)v_i + u_i^{\top}X(dv_i) .
$$

*Step 2 — kill the two outer terms.* $Xv_i = \sigma_iu_i$ and $X^{\top}u_i = \sigma_iv_i$, so those
terms are $\sigma_i(du_i)^{\top}u_i$ and $\sigma_iv_i^{\top}(dv_i)$. Differentiating
$u_i^{\top}u_i = 1$ gives $(du_i)^{\top}u_i = 0$, and likewise for $v_i$. Both vanish.

Distinct singular values are what make $u_i, v_i$ locally differentiable functions of $X$, which
is where the hypothesis is used.

*Step 3 — sum.* Therefore $d\sigma_i = u_i^{\top}(dX)v_i = \operatorname{tr}\bigl( (u_iv_i^{\top})^{\top}dX \bigr)$,
and summing over $i$,

$$
d\lVert X \rVert_{*} = \operatorname{tr}\Bigl( \bigl( \textstyle\sum_i u_iv_i^{\top} \bigr)^{\top}dX \Bigr)
= \operatorname{tr}\bigl( (UV^{\top})^{\top}dX \bigr) .
$$

$$
\boxed{\nabla_X \lVert X \rVert_{*} = UV^{\top}}
$$

**Key takeaway.** When a singular value hits zero the norm stops being differentiable and
$UV^{\top}$ becomes one element of the subdifferential — which is the whole point in low-rank
matrix completion, where the optimum is rank-deficient.

In [44]:
Xn = rng.standard_normal((6, 4))
Un, sn, Vtn = np.linalg.svd(Xn, full_matrices=False)
G = Un @ Vtn
nuc = lambda Z: np.linalg.svd(Z, compute_uv=False).sum()
h = 1e-6
fd = np.zeros_like(Xn)
for idx in np.ndindex(Xn.shape):
    E = np.zeros_like(Xn)
    E[idx] = h
    fd[idx] = (nuc(Xn + E) - nuc(Xn - E)) / (2 * h)
print("singular values:", sn)
print("max |U V^T - finite difference| =", np.abs(fd - G).max())
print("U V^T has orthonormal columns:", np.abs(G.T @ G - np.eye(4)).max())
print("nuclear norm =", nuc(Xn), " = tr((X^T X)^(1/2)) =",
      np.trace(sla.sqrtm(Xn.T @ Xn)).real)
assert np.abs(fd - G).max() < 1e-6

singular values: [3.6346 2.4508 1.443  1.2628]
max |U V^T - finite difference| = 1.5306132472581169e-09
U V^T has orthonormal columns: 8.881784197001252e-16
nuclear norm = 8.791219217082961  = tr((X^T X)^(1/2)) = 8.791219217082965


### Problem L3.5 — Mixing rate of a random walk

**Statement.** Let $G$ be connected, non-bipartite and unweighted, and let $M = AD^{-1}$ be the
column-stochastic random-walk operator with stationary vector $\pi = d/(2\lvert E \rvert)$. Prove

$$
\lVert M^{k}x_0 - \pi \rVert_{\pi^{-1}} \ \le \ \lambda_{\star}^{k}\,\lVert x_0 - \pi \rVert_{\pi^{-1}},
\qquad \lVert y \rVert_{\pi^{-1}}^2 = \sum_i \frac{y_i^2}{\pi_i},
$$

where $\lambda_{\star}$ is the second-largest eigenvalue modulus of
$\mathcal{A} = D^{-1/2}AD^{-1/2}$.

**Intuition.** $M$ is not symmetric, but it is similar to the symmetric $\mathcal{A}$, and the
weighted norm is exactly the Euclidean norm in the coordinates where that similarity happens.

**Solution.**

*Step 1 — stationarity.* $M\pi = AD^{-1}d/(2\lvert E \rvert) = A\mathbf{1}/(2\lvert E \rvert) = d/(2\lvert E \rvert) = \pi$.

*Step 2 — symmetrize.* $M = D^{1/2}\mathcal{A}D^{-1/2}$, so $D^{-1/2}M^{k} = \mathcal{A}^{k}D^{-1/2}$.

*Step 3 — the weighted norm is Euclidean after rescaling.* With $y = x - \pi$ and $z = D^{-1/2}y$,

$$
\lVert y \rVert_{\pi^{-1}}^2 = \sum_i \frac{y_i^2}{d_i/(2\lvert E \rvert)} = 2\lvert E \rvert\,\lVert z \rVert_2^2 .
$$

*Step 4 — orthogonality to the top eigenvector.* $\mathcal{A}$ is symmetric with
$\mathcal{A}(D^{1/2}\mathbf{1}) = D^{1/2}\mathbf{1}$, and

$$
(D^{1/2}\mathbf{1})^{\top}z = \mathbf{1}^{\top}(x_0 - \pi) = 1 - 1 = 0 ,
$$

so $z$ lies in the invariant orthogonal complement of the Perron direction.

*Step 5 — contract.* On that complement the spectral theorem gives
$\lVert \mathcal{A}^{k}z \rVert_2 \le \lambda_{\star}^{k}\lVert z \rVert_2$, and connectivity plus
non-bipartiteness give $\lambda_{\star} \lt 1$. Undoing Step 3 restores the weighted norm.

$$
\boxed{\lVert M^{k}x_0 - \pi \rVert_{\pi^{-1}} \le \lambda_{\star}^{k}\lVert x_0 - \pi \rVert_{\pi^{-1}}, \qquad t_{\mathrm{mix}} \sim \frac{1}{1 - \lambda_{\star}}}
$$

**Key takeaway.** Bipartiteness puts an eigenvalue at $-1$ and destroys convergence, exactly as
the two-state swap of Section 7.3 does; the spectral gap $1 - \lambda_{\star}$ is the mixing rate.

In [45]:
Amx = np.array([[0.0, 1, 1, 0, 0], [1, 0, 1, 1, 0], [1, 1, 0, 1, 1],
                [0, 1, 1, 0, 1], [0, 0, 1, 1, 0]])
dmx = Amx.sum(1)
Mmx = Amx @ np.diag(1.0 / dmx)
pi = dmx / dmx.sum()
Acal = np.diag(dmx ** -0.5) @ Amx @ np.diag(dmx ** -0.5)
mu = np.linalg.eigvalsh(Acal)
lam_star = max(abs(mu[-2]), abs(mu[0]))
wnorm = lambda y: np.sqrt(np.sum(y ** 2 / pi))
x0 = np.zeros(5)
x0[0] = 1.0
print("column sums of M:", Mmx.sum(axis=0), "   ||M pi - pi|| =", np.linalg.norm(Mmx @ pi - pi))
print("spectrum of D^-1/2 A D^-1/2:", mu, "   lambda_star =", lam_star)
xk = x0.copy()
for k in range(1, 9):
    xk = Mmx @ xk
    lhs = wnorm(xk - pi)
    print(f"  k={k}  weighted error {lhs:.6e}   bound {lam_star**k * wnorm(x0 - pi):.6e}")
    assert lhs <= lam_star ** k * wnorm(x0 - pi) + 1e-12
Abip = np.array([[0.0, 1], [1.0, 0.0]])
mub = np.linalg.eigvalsh(np.diag(Abip.sum(1) ** -0.5) @ Abip @ np.diag(Abip.sum(1) ** -0.5))
print("bipartite two-cycle: spectrum", mub, " -> lambda_star = 1, no contraction")
assert abs(max(abs(mub[0]), abs(mub[-2])) - 1.0) < 1e-12

column sums of M: [1. 1. 1. 1. 1.]    ||M pi - pi|| = 0.0
spectrum of D^-1/2 A D^-1/2: [-0.6076 -0.5    -0.1667  0.2743  1.    ]    lambda_star = 0.6076252185107652
  k=1  weighted error 1.020621e+00   bound 1.488372e+00
  k=2  weighted error 5.215273e-01   bound 9.043722e-01
  k=3  weighted error 2.883966e-01   bound 5.495194e-01
  k=4  weighted error 1.647453e-01   bound 3.339018e-01
  k=5  weighted error 9.582759e-02   bound 2.028872e-01
  k=6  weighted error 5.643559e-02   bound 1.232794e-01
  k=7  weighted error 3.353893e-02   bound 7.490765e-02
  k=8  weighted error 2.006403e-02   bound 4.551578e-02
bipartite two-cycle: spectrum [-1.  1.]  -> lambda_star = 1, no contraction


### Problem L3.6 — Laplacian of a Cartesian product graph

**Statement.** For graphs $G_1$, $G_2$ with Laplacians $L_1 \in \mathbb{R}^{n_1 \times n_1}$,
$L_2 \in \mathbb{R}^{n_2 \times n_2}$, prove that the Cartesian product $G_1 \square G_2$ has
Laplacian $L = L_1 \otimes I_{n_2} + I_{n_1}\otimes L_2$, and that its spectrum is
$\lbrace \mu_i + \nu_j \rbrace$.

**Intuition.** In the product graph a vertex moves in one coordinate at a time, so the two
Laplacians act on independent factors and simply add.

**Solution.**

*Step 1 — adjacency.* By definition $(u_1, u_2)$ and $(v_1, v_2)$ are adjacent in the product
exactly when either $u_1 = v_1$ and $u_2 \sim v_2$, or $u_2 = v_2$ and $u_1 \sim v_1$. In the
ordering used by $\otimes$ this is $A = A_1 \otimes I_{n_2} + I_{n_1}\otimes A_2$.

*Step 2 — degrees.* The degree of $(u_1, u_2)$ is $d^{(1)}_{u_1} + d^{(2)}_{u_2}$, so
$D = D_1 \otimes I_{n_2} + I_{n_1}\otimes D_2$.

*Step 3 — subtract.* $L = D - A = (D_1 - A_1)\otimes I_{n_2} + I_{n_1}\otimes(D_2 - A_2)$.

*Step 4 — spectrum.* Let $L_1u_i = \mu_iu_i$ and $L_2v_j = \nu_jv_j$ with orthonormal
eigenvectors. Using $(P \otimes Q)(x \otimes y) = Px \otimes Qy$,

$$
L(u_i \otimes v_j) = (\mu_i + \nu_j)(u_i \otimes v_j),
$$

and the $n_1n_2$ vectors $u_i \otimes v_j$ are orthonormal, hence a full eigenbasis.

$$
\boxed{L_{G_1 \square G_2} = L_1 \otimes I + I \otimes L_2, \qquad \operatorname{spec} = \lbrace \mu_i + \nu_j \rbrace}
$$

**Key takeaway.** Grids, hypercubes and tori are Cartesian products of paths and cycles, so their
whole spectra are known in closed form — which is why they are the standard test problems in
spectral graph theory.

In [46]:
def path_lap(k):
    Ap = np.zeros((k, k))
    for i in range(k - 1):
        Ap[i, i + 1] = Ap[i + 1, i] = 1.0
    return np.diag(Ap.sum(1)) - Ap, Ap


L1, A1 = path_lap(3)
L2, A2 = path_lap(4)
n1, n2 = 3, 4
Aprod = np.kron(A1, np.eye(n2)) + np.kron(np.eye(n1), A2)
Lprod = np.diag(Aprod.sum(1)) - Aprod
Lkron = np.kron(L1, np.eye(n2)) + np.kron(np.eye(n1), L2)
mu = np.linalg.eigvalsh(L1)
nu = np.linalg.eigvalsh(L2)
pred = np.sort(np.array([a + b for a in mu for b in nu]))
print("||L(product) - (L1 kron I + I kron L2)||_F =", np.linalg.norm(Lprod - Lkron))
print("spectrum of the 3x4 grid :", np.round(np.linalg.eigvalsh(Lprod), 6))
print("predicted mu_i + nu_j    :", np.round(pred, 6))
assert np.linalg.norm(Lprod - Lkron) < 1e-12
assert np.allclose(np.linalg.eigvalsh(Lprod), pred)

||L(product) - (L1 kron I + I kron L2)||_F = 0.0
spectrum of the 3x4 grid : [-0.      0.5858  1.      1.5858  2.      3.      3.      3.4142  3.5858
  4.4142  5.      6.4142]
predicted mu_i + nu_j    : [0.     0.5858 1.     1.5858 2.     3.     3.     3.4142 3.5858 4.4142
 5.     6.4142]


### Problem L3.7 — The graph Dirac operator

**Statement.** Let $B \in \mathbb{R}^{n \times m}$ be an oriented incidence matrix and define

$$
K = \begin{pmatrix} 0 & B \\ B^{\top} & 0 \end{pmatrix} \in \mathbb{R}^{(n+m)\times(n+m)} .
$$

Compute $K^2$ and interpret its diagonal blocks. Deduce that the non-zero spectra of $BB^{\top}$
and $B^{\top}B$ coincide.

**Intuition.** $K$ is a square root of a block-diagonal Laplacian: it moves signals between
vertices and edges, and doing it twice returns to where it started.

**Solution.**

*Step 1.* Block multiplication:

$$
K^2 = \begin{pmatrix} BB^{\top} & 0 \\ 0 & B^{\top}B \end{pmatrix} .
$$

*Step 2.* The upper block is the vertex Laplacian $L_0 = BB^{\top} = D - A$ by Problem L1.10; the
lower block $L_1 = B^{\top}B$ is the edge Laplacian, which governs circulations on edges.

*Step 3.* $K$ is symmetric, so its spectrum is real and $\operatorname{spec}(K^2) = \lbrace \kappa^2 : \kappa \in \operatorname{spec}(K) \rbrace$.
Since $K$ is also block-anti-diagonal, $\kappa \in \operatorname{spec}(K)$ implies
$-\kappa \in \operatorname{spec}(K)$: conjugating by $\operatorname{diag}(I_n, -I_m)$ sends
$K$ to $-K$.

*Step 4.* Hence the eigenvalues of $K^2$ come in matched pairs, one contributed by each block, so
$BB^{\top}$ and $B^{\top}B$ have the same non-zero eigenvalues with the same multiplicities.

$$
\boxed{K^2 = \begin{pmatrix} L_0 & 0 \\ 0 & L_1 \end{pmatrix}, \qquad \operatorname{spec}(L_0)\setminus\lbrace 0 \rbrace = \operatorname{spec}(L_1)\setminus\lbrace 0 \rbrace}
$$

**Key takeaway.** This is the graph version of the fact that $\lVert X \rVert$ can be read from
either $XX^{\top}$ or $X^{\top}X$; the operator $K$ is the discrete Dirac operator whose square is
the Hodge Laplacian.

In [47]:
Ad2 = np.array([[0.0, 1, 1, 0], [1, 0, 1, 1], [1, 1, 0, 0], [0, 1, 0, 0]])
ed = [(i, j) for i in range(4) for j in range(i + 1, 4) if Ad2[i, j] > 0]
Bd = np.zeros((4, len(ed)))
for k, (i, j) in enumerate(ed):
    Bd[i, k], Bd[j, k] = 1.0, -1.0
nB, mB = Bd.shape
Kd = np.block([[np.zeros((nB, nB)), Bd], [Bd.T, np.zeros((mB, mB))]])
K2 = Kd @ Kd
L0 = Bd @ Bd.T
L1 = Bd.T @ Bd
print("n =", nB, " m =", mB)
print("||K^2 - blockdiag(L0, L1)||_F =",
      np.linalg.norm(K2 - np.block([[L0, np.zeros((nB, mB))], [np.zeros((mB, nB)), L1]])))
print("spec(L0):", np.round(np.linalg.eigvalsh(L0), 6))
print("spec(L1):", np.round(np.linalg.eigvalsh(L1), 6))
print("spec(K) :", np.round(np.linalg.eigvalsh(Kd), 6), " -> symmetric about zero")
nz0 = np.sort(np.linalg.eigvalsh(L0)[np.linalg.eigvalsh(L0) > 1e-10])
nz1 = np.sort(np.linalg.eigvalsh(L1)[np.linalg.eigvalsh(L1) > 1e-10])
assert np.allclose(nz0, nz1)
assert np.allclose(np.sort(np.linalg.eigvalsh(Kd)), np.sort(-np.linalg.eigvalsh(Kd)))

n = 4  m = 4
||K^2 - blockdiag(L0, L1)||_F = 0.0
spec(L0): [-0.  1.  3.  4.]
spec(L1): [-0.  1.  3.  4.]
spec(K) : [-2.     -1.7321 -1.     -0.      0.      1.      1.7321  2.    ]  -> symmetric about zero


### Problem L3.8 — Implicit differentiation of an equilibrium graph network

**Statement.** A deep equilibrium GNN defines $H^{\star}$ implicitly by
$H^{\star} = \sigma(\hat{A}H^{\star}W + X)$. Assuming the linearized map is invertible, derive
$\nabla_W J$ without unrolling the fixed-point iteration.

**Intuition.** Differentiate the fixed-point equation itself; the implicit function theorem
replaces the whole unrolled chain with one linear solve.

**Solution.**

*Step 1 — differentiate the fixed point.* Put $Z^{\star} = \hat{A}H^{\star}W + X$ and
$S = \sigma'(Z^{\star})$. Then

$$
dH^{\star} = S \odot \bigl( \hat{A}(dH^{\star})W + \hat{A}H^{\star}\,dW \bigr) .
$$

*Step 2 — isolate.* Define the linear operator $\mathcal{J}(\Delta) = S \odot (\hat{A}\Delta W)$.
The equation reads

$$
(\mathcal{I} - \mathcal{J})(dH^{\star}) = S \odot \bigl( \hat{A}H^{\star}\,dW \bigr) .
$$

Invertibility of $\mathcal{I} - \mathcal{J}$ is the hypothesis; it holds whenever the fixed-point
map is a contraction.

*Step 3 — adjoint.* We want $dJ = \langle \nabla_{H^{\star}}J, dH^{\star} \rangle_F$. Solve the
adjoint system for $Y^{\star}$:

$$
(\mathcal{I} - \mathcal{J}^{\ast})(Y^{\star}) = \nabla_{H^{\star}}J,
\qquad \mathcal{J}^{\ast}(Y) = \hat{A}^{\top}\bigl( Y \odot S \bigr)W^{\top},
$$

the adjoint being taken in the Frobenius inner product.

*Step 4 — substitute.* Then
$dJ = \langle Y^{\star}, S \odot (\hat{A}H^{\star}dW) \rangle_F = \langle (\hat{A}H^{\star})^{\top}(Y^{\star}\odot S), dW \rangle_F$.

$$
\boxed{\text{solve } (\mathcal{I} - \mathcal{J}^{\ast})(Y^{\star}) = \nabla_{H^{\star}}J, \quad \text{then } \nabla_W J = (\hat{A}H^{\star})^{\top}\bigl( Y^{\star} \odot \sigma'(Z^{\star}) \bigr)}
$$

**Key takeaway.** Memory cost is one linear solve instead of one stored activation per iteration,
so an equilibrium network has the memory footprint of a single layer at any effective depth.

In [48]:
ne, de = 5, 3
Ae = np.array([[0.0, 1, 0, 0, 1], [1, 0, 1, 0, 0], [0, 1, 0, 1, 0],
               [0, 0, 1, 0, 1], [1, 0, 0, 1, 0]])
At = Ae + np.eye(ne)
dt_ = At.sum(1)
Ah = np.diag(dt_ ** -0.5) @ At @ np.diag(dt_ ** -0.5)
We = rng.standard_normal((de, de)) * 0.25
Xe = rng.standard_normal((ne, de))
Te = rng.standard_normal((ne, de))


def equilibrium(W_, iters=4000):
    H = np.zeros((ne, de))
    for _ in range(iters):
        H = np.tanh(Ah @ H @ W_ + Xe)
    return H


def loss_eq(W_):
    return 0.5 * np.linalg.norm(equilibrium(W_) - Te) ** 2


Hs = equilibrium(We)
Zs = Ah @ Hs @ We + Xe
S = 1 - np.tanh(Zs) ** 2
Jop = np.kron(We.T, Ah) * S.reshape(-1, order="F")[:, None]      # vec form of Delta -> S * (A Delta W)
gH = Hs - Te
Yv = np.linalg.solve(np.eye(ne * de) - Jop.T, vec(gH))
Ys = Yv.reshape(ne, de, order="F")
gW = (Ah @ Hs).T @ (Ys * S)
h = 1e-5
fd = np.zeros_like(We)
for idx in np.ndindex(We.shape):
    E = np.zeros_like(We)
    E[idx] = h
    fd[idx] = (loss_eq(We + E) - loss_eq(We - E)) / (2 * h)
print("||H* - sigma(A H* W + X)||_F =", np.linalg.norm(Hs - np.tanh(Ah @ Hs @ We + Xe)))
print("implicit gradient:\n", gW)
print("finite difference:\n", fd)
print("max difference   :", np.abs(fd - gW).max())
assert np.linalg.norm(Hs - np.tanh(Ah @ Hs @ We + Xe)) < 1e-10
assert np.abs(fd - gW).max() < 1e-6

||H* - sigma(A H* W + X)||_F = 2.1138016631127878e-16
implicit gradient:
 [[ 0.2394 -0.001  -0.2487]
 [ 0.1551 -0.0813 -0.342 ]
 [-0.0538 -0.0389 -0.1391]]
finite difference:
 [[ 0.2394 -0.001  -0.2487]
 [ 0.1551 -0.0813 -0.342 ]
 [-0.0538 -0.0389 -0.1391]]
max difference   : 7.314586436546477e-11


### Problem L3.9 — Strict saddle property of matrix factorization

**Statement.** Let $f(U, V) = \tfrac12\lVert UV^{\top} - M \rVert_F^2$ with
$U \in \mathbb{R}^{d \times r}$, $V \in \mathbb{R}^{k \times r}$. Suppose $(U,V)$ is a critical
point with $R = UV^{\top} - M \neq 0$ and with a common right null vector, that is
$w \in \mathbb{R}^{r}$, $\lVert w \rVert_2 = 1$, $Uw = 0$ and $Vw = 0$. Prove that the Hessian of
$f$ at $(U,V)$ has a strictly negative eigenvalue.

**Intuition.** The shared null direction is unused capacity; pushing the top residual direction
into it lowers the loss at second order, with no first-order cost.

**Solution.**

*Step 1 — pick the escape direction.* Let $\sigma = \sigma_1(R) \gt 0$ with unit singular vectors
$u, v$, so $Rv = \sigma u$ and $u^{\top}Rv = \sigma$. Set

$$
\Delta U = u\,w^{\top}, \qquad \Delta V = -\,v\,w^{\top} .
$$

*Step 2 — the perturbed product is exact.* Expanding and using $Uw = Vw = 0$ and
$w^{\top}w = 1$,

$$
(U + \epsilon\Delta U)(V + \epsilon\Delta V)^{\top}
= UV^{\top} - \epsilon (Uw)v^{\top} + \epsilon u(Vw)^{\top} - \epsilon^{2}uv^{\top}
= UV^{\top} - \epsilon^{2}uv^{\top} .
$$

*Step 3 — evaluate $f$ along the curve.* With $\lVert uv^{\top} \rVert_F = 1$ and
$\langle R, uv^{\top} \rangle_F = u^{\top}Rv = \sigma$,

$$
f(U + \epsilon\Delta U, V + \epsilon\Delta V) = \tfrac12\lVert R - \epsilon^{2}uv^{\top} \rVert_F^2
= f(U,V) - \sigma\epsilon^{2} + \tfrac12\epsilon^{4} .
$$

*Step 4 — read off the curvature.* $(U,V)$ is critical, so the first-order term of the Taylor
expansion vanishes and the $\epsilon^2$ coefficient is $\tfrac12\nabla^2 f[\Delta]$. Comparing,

$$
\nabla^2 f\bigl[ (\Delta U, \Delta V) \bigr] = -2\sigma \ \lt \ 0 .
$$

$$
\boxed{\nabla^2 f\bigl[ (uw^{\top}, -vw^{\top}) \bigr] = -2\sigma_1(UV^{\top} - M) \lt 0}
$$

**Key takeaway.** Every non-global critical point with unused rank is a strict saddle, so
perturbed gradient descent escapes it — the standard justification for optimizing a factorized
model directly.

In [49]:
d_, k_, r_ = 5, 4, 3
Mt = rng.standard_normal((d_, 2)) @ rng.standard_normal((2, k_))    # rank 2 target
Ub = np.zeros((d_, r_))
Vb = np.zeros((k_, r_))
Ub[:, 0] = rng.standard_normal(d_)
Vb[:, 0] = rng.standard_normal(k_)
for _ in range(4000):                     # alternating least squares on the first column only
    Ub[:, :1] = Mt @ Vb[:, :1] @ np.linalg.inv(Vb[:, :1].T @ Vb[:, :1])
    Vb[:, :1] = Mt.T @ Ub[:, :1] @ np.linalg.inv(Ub[:, :1].T @ Ub[:, :1])
f = lambda U_, V_: 0.5 * np.linalg.norm(U_ @ V_.T - Mt) ** 2
R = Ub @ Vb.T - Mt
gU, gV = R @ Vb, R.T @ Ub
print("||grad_U|| =", np.linalg.norm(gU), "   ||grad_V|| =", np.linalg.norm(gV))
w = np.zeros(r_)
w[2] = 1.0
print("U w =", Ub @ w, "   V w =", Vb @ w)
Us, ss, Vts = np.linalg.svd(R)
u, v, sig = Us[:, 0], Vts[0], ss[0]
dU, dV = np.outer(u, w), -np.outer(v, w)
print("sigma_1(R) =", sig)
for eps in (1e-1, 1e-2, 1e-3):
    val = f(Ub + eps * dU, Vb + eps * dV)
    print(f"  eps={eps:.0e}   f = {val:.10f}   f0 - sigma eps^2 + eps^4/2 = "
          f"{f(Ub, Vb) - sig * eps**2 + 0.5 * eps**4:.10f}")
curv = (f(Ub + 1e-3 * dU, Vb + 1e-3 * dV) - 2 * f(Ub, Vb)
        + f(Ub - 1e-3 * dU, Vb - 1e-3 * dV)) / 1e-6
print(f"second directional derivative = {curv:.6f}   predicted -2 sigma = {-2*sig:.6f}")
assert np.linalg.norm(gU) < 1e-8 and np.linalg.norm(gV) < 1e-8
assert curv < 0 and abs(curv + 2 * sig) < 1e-3

||grad_U|| = 3.554916897103672e-16    ||grad_V|| = 8.344366497024685e-16
U w = [0. 0. 0. 0. 0.]    V w = [0. 0. 0. 0.]
sigma_1(R) = 2.2193613441798816
  eps=1e-01   f = 2.4406387746   f0 - sigma eps^2 + eps^4/2 = 2.4406387746
  eps=1e-02   f = 2.4625604569   f0 - sigma eps^2 + eps^4/2 = 2.4625604569
  eps=1e-03   f = 2.4627801687   f0 - sigma eps^2 + eps^4/2 = 2.4627801687
second directional derivative = -4.438722   predicted -2 sigma = -4.438723
